# Import and Settings

In [1]:
import solara
import solara.lab
import os
import time
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import matplotlib.dates as mdates
import matplotlib.cm as cm
import matplotlib.colors as colors 
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from branca.colormap import LinearColormap, linear
plt.close('all')
from xugrid.plot import line
import xarray as xr
import pandas as pd
import numpy as np
from ipyleaflet import GeoJSON, Map, basemaps, Polyline, Rectangle, CircleMarker, LayerGroup, LegendControl, AntPath, WidgetControl, AwesomeIcon, Marker, Icon, DrawControl, LayersControl
from shapely.geometry import LineString, shape
import ipywidgets as widgets
import datetime
from pathlib import Path
import geopandas as gpd
import json
import folium
import subprocess
import glob
import shutil
import cartopy.crs as ccrs
proj = ccrs.PlateCarree()
import cartopy.io.img_tiles as cimgt
from pyproj import Transformer
import io
import contextlib
import threading 

import dfm_tools as dfmt
import hydrolib.core.dflowfm as hcdfm 
import hydromt
from hydromt import DataCatalog # from hydromt.data_catalog.data_catalog import DataCatalog
from hydromt_sfincs import SfincsModel
from hydromt_sfincs.workflows import river_source_points
from hydromt_fiat.fiat import FIATModel#, fetch_data

# from hydromt.log import setuplog # xxxxxxxxxxxxxxxxxxxxx

m_SFINCS = Map(center=(53.5, -0.5), zoom=8, scroll_wheel_zoom=True,basemap=basemaps.OpenStreetMap.Mapnik)
m_DHYDRO = Map(center=(53.5, -0.5), zoom=8, scroll_wheel_zoom=True,basemap=basemaps.OpenStreetMap.Mapnik)
m_FIAT = Map(center=(53.5, -0.5), zoom=8, scroll_wheel_zoom=True,basemap=basemaps.OpenStreetMap.Mapnik)

# SFINCS

## 0) Import & Settings

In [2]:
TAB_NAMES_SFINCS = ["Config", "ModDom","Elev","ActC","wlBnd","rInflP","LRoughInf","subgrid","obs","forcing","model"] # ,"GridGen"
tab_layers_SFINCS = {name: LayerGroup() for name in TAB_NAMES_SFINCS}

current_layer_group_SFINCS = solara.reactive(tab_layers_SFINCS['Config'])
tab_controls_SFINCS = solara.reactive({})
control_update_signal_SFINCS = solara.reactive(0)
current_control_SFINCS = solara.reactive(None)
rectangle_SFINCS = solara.reactive(None)

control_signals_SFINCS = {name: solara.reactive(0) for name in TAB_NAMES_SFINCS}

configdone_SFINCS = solara.reactive(False) 
modeldomaindone_SFINCS = solara.reactive(False) 
elevationdone_SFINCS = solara.reactive(False) 
activecellsdone_SFINCS = solara.reactive(False) 
waterlevelbnddone_SFINCS = solara.reactive(False) 
riverinflowdone_SFINCS = solara.reactive(False) 
roughnesssubgriddone_SFINCS = solara.reactive(False) 
obsdone_SFINCS = solara.reactive(False)
forcingdone_SFINCS = solara.reactive(False)

selected_tab_SFINCS = solara.reactive('Config') 
continuous_update_SFINCS = solara.reactive(True)

## 1) Configuration

In [3]:
sf_logger_name = solara.reactive("SFINCS_log")

model_path = solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\SFINCS\\models\\test")# ("C:\\") # HIER PATH ÄNDERN
sf_root = model_path # HERE THE SAME, DELETED v1 FOLDER
model_name_SFINCS = solara.reactive(model_path.value.split("\\")[-1]) # get model_name from model_path

# modelbuilder domain geojson
region_fn = solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\SFINCS\\data\\sfincs_Humber_modelbuilder_domain.geojson")# ("C:\\") # HIER PATH ÄNDERN 

init_tools_status = solara.reactive("Idle")
def init_tools():
    global data_catalog, sf
    init_tools_status.set("Plotting")
    # sf_logger = setuplog(sf_logger_name.value,log_level=10)
    # Use line below for own data_catalog file
    # data_catalog = hydromt.DataCatalog(data_libs=data_catalog_fn,logger=sf_logger)
    # Use line below for 'deltares_data' data_catalog file
    data_catalog = hydromt.DataCatalog(data_libs=["deltares_data"])#,logger=sf_logger)
    sf = SfincsModel(data_libs=["deltares_data"], root=sf_root.value, mode='w+')

    init_tools_status.set("Done")

    configdone_SFINCS.set(True) 

data_dict = {
    'topo': 'fabdem',
    'bathy': 'gebco',
    'infiltration': 'gcn250',
    'lulc': 'globcover',
    'basins': 'hydro_basin_atlas_level12',
    "hydrography": "merit_hydro",
    'precip': 'era5_hourly'
}

In [4]:
@solara.component
def Tab_SFINCS_Configuration():
    
    with solara.Card("Configuration", style={"width": "100%", "padding": "10px"}):

        solara.Markdown("**Select Model:**",style={"color": "inherit"})
        solara.InputText("Model directory", value=model_path, continuous_update=True, style={"marginBottom": "20px"})
        solara.Text(f"Model Name: {model_name_SFINCS.value}", style={"marginBottom": "50px"})
        # solara.InputText("Model Name (avoid spaces)", value=model_name_SFINCS, continuous_update=continuous_update_SFINCS.value)
        path_model = Path(model_path.value)
        if model_path.value:
            if not path_model.exists():
                solara.Error("Path does not exist.")

        solara.Markdown("**Select modelbuilder domain:**",style={"color": "inherit"})
        solara.InputText("Select .geojson-file", value=region_fn, continuous_update=True, style={"marginBottom": "20px"})
        path_geojson = Path(region_fn.value)
        if region_fn.value:
            if not region_fn.value.endswith(".geojson"):
                solara.Error(label='Error: File must end with ".geojson"', text=False, dense=True, outlined=True, icon=False)
            elif not path_geojson.exists():
                solara.Error("File does not exist.")

        solara.Button(label="Initialise Tools",on_click=init_tools, continuous_update=True)

        if init_tools_status.value == "Plotting":
            solara.Markdown("Initialising Tools... Please wait.",style={"color": "inherit"})
        elif init_tools_status.value == "Done":
            solara.Markdown("**Done!**",style={"color": "inherit"})

    with solara.Row(justify="end"):     
        solara.Button(label="Go to Step 2.", on_click=lambda: selected_tab_SFINCS.set('ModDom'),disabled=not configdone_SFINCS.value)


# Tab_SFINCS_Configuration()


## 2) Model Domain (including Grid Generation)

Configure the model domain based on geojson file given in `region_fn`

In [5]:

setup_domain_status = solara.reactive("Idle")
ref_date = solara.reactive(datetime.date(2010, 2, 1))
date_min = solara.reactive(datetime.date(2010, 2, 5))
date_max = solara.reactive(datetime.date(2010, 2, 7))

def set_up_domain():
    global SW, NE, rectangle_SFINCS, ds_basins, sf
    setup_domain_status.set("Plotting")
    
    region = gpd.read_file(region_fn.value)

    sf.config.update(        # UPDATE # sf.setup_config( 
        **{
            "tref": ref_date.value.strftime("%Y%m%d 000000"),
            "tstart": date_min.value.strftime("%Y%m%d 000000"),
            "tstop": date_max.value.strftime("%Y%m%d 000000"),
        }
    )
    ds_basins = data_catalog.get_geodataframe(data_dict['basins'],geom=region)
    geo_json_data = json.loads(ds_basins.to_json())
    geo_json_layer = GeoJSON(data=geo_json_data)
    
    SW = (ds_basins['geometry'].total_bounds[[0,2,1,3]][2], ds_basins['geometry'].total_bounds[[0,2,1,3]][0])
    NE = (ds_basins['geometry'].total_bounds[[0,2,1,3]][3], ds_basins['geometry'].total_bounds[[0,2,1,3]][1])
        
    rectangle_SFINCS.value = Rectangle(bounds=(SW, NE), color="white", fill_opacity=0) # , weight=1)

    if rectangle_SFINCS.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(rectangle_SFINCS.value)
    current_layer_group_SFINCS.value.add_layer(rectangle_SFINCS.value)

    if geo_json_layer in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(geo_json_layer)
    current_layer_group_SFINCS.value.add_layer(geo_json_layer)

    sf.grid.create_from_region(                                 # UPDATE # sf.setup_grid_from_region()
        region={"bbox": ds_basins['geometry'].total_bounds},
        res=200,
        rotated=False,
        crs="utm"
    )

    setup_domain_status.set("Done")

    modeldomaindone_SFINCS.set(True) 
    

In [6]:
@solara.component
def Tab_SFINCS_Model_Domain():
    solara.Markdown("Date selection:",style={"color": "inherit"})
    solara.Text("Reference date:"); solara.lab.InputDate(ref_date, style={"marginBottom": "20px"})
    solara.Text("Select start date:"); solara.lab.InputDate(date_min, style={"marginBottom": "20px"})
    solara.Text("Select stop date:"); solara.lab.InputDate(date_max, style={"marginBottom": "20px"})
    if date_max.value < date_min.value:
        solara.Markdown("**Warning**: The end date cannot be earlier than the start date.", 
                        style={"color": "red"})
    
    # solara.use_effect(update_rectangle)#, dependencies=[lat_min.value, lat_max.value, lon_min.value, lon_max.value]) # RESET THE RECTANGLE: Model area
    solara.Button(label="set-up domain",on_click=set_up_domain, continuous_update=True)
    if setup_domain_status.value == "Plotting":
        solara.Markdown("Set-up model domain... Please wait.",style={"color": "inherit"})
    elif setup_domain_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})

    with solara.Row(justify="end"):
        solara.Button(label="Go to Step 3.", on_click=lambda: selected_tab_SFINCS.set('Elev'),disabled=not modeldomaindone_SFINCS.value)
    
# Tab_SFINCS_Model_Domain()

## 3) Grid Generation

Implemented in `5) Active Cells`



In Vorlage steht:



Next we build the model grid in our domain. This will start as the bounding box of the domain, but we will refine this later by specifying active cells. Any input we provide will automatically be converted to the `crs` we specify here. The resolution `res` is in units of the `crs`.

In [7]:
# # # def grid_generation():
# # #     global sf
# # #     sf.setup_grid_from_region(
# # #         region={"bbox": ds_basins['geometry'].total_bounds},
# # #         res=200,
# # #         rotated=False,
# # #         crs="utm"
# # #     )

# # #     datasets_dep = [{'elevtn': data_dict['topo'], 'zmin': 0.001}, {'elevtn': data_dict['bathy']}]
# # #     _ = sf.setup_dep(datasets_dep=datasets_dep)
    
# # #     # Convert resulting grid lines to GeoDataFrame
# # #     grid_lines = sf.grid.to_geodataframe_lines()
# # #     grid_json = json.loads(grid_lines.to_json())

# # #     # Add as GeoJSON layer to Solara map
# # #     grid_layer = GeoJSON(data=grid_json, style={"color": "blue", "weight": 1}, name="Grid")
    
# # #     if grid_layer in current_layer_group_SFINCS.value.layers:
# # #         current_layer_group_SFINCS.value.remove_layer(grid_layer)
# # #     current_layer_group_SFINCS.value.add_layer(grid_layer)

# # # ###############################
# # # @solara.component
# # # def Tab_SFINCS_Grid_Generation():
# # #    solara.Button(label="Grid Generation",on_click=grid_generation, continuous_update=True)
    
# # # # Tab_SFINCS_Grid_Generation()

## 4) Elevation

Refining the grid is generally done based on elevation data, so first we need to load those in.


Here we use global data sets, Copernicus DEM for topography (resolution 1arcsec ~ 30m) and GEBCO (resolution 450m) for bathymetry. When available for your region we highly recommend using more high resolution data (in this case particularly for bathymetry)

In [8]:
step_elev = solara.reactive(7)
elev_layer = solara.reactive(LayerGroup()) 

elev_plot_done_SFINCS = solara.reactive(False) 

plot_elev_status = solara.reactive("Idle")
def plot_elev():
    global scatter_layer_g, legend_elev# , elev_layer
    plot_elev_status.set("Plotting")
    bounds_array = np.array(ds_basins.total_bounds, dtype=float)
    bbox = tuple(bounds_array.tolist())
    # f = sf.data_catalog.get_rasterdataset("fabdem", bbox=bbox) # topography # PROBLEM: min = -9999
    g = sf.data_catalog.get_rasterdataset("gebco", bbox=bbox) # bathymetry
    
    terrain = cm.get_cmap("terrain", 256)
    terrain_colors = terrain(np.linspace(0, 1, 256))
    land_part = terrain_colors[70:] ########### -----------------------------------------> ggf anheben!!!
    land_rescaled = land_part[np.linspace(0, len(land_part) - 1, 128, dtype=int)]
    
    mid_blue = np.array([70, 161, 230]) / 255  # steelblue
    dark_blue = np.array([41, 50, 141]) / 255    # navy
    blue_rgb = np.linspace(dark_blue, mid_blue, 128)
    blue_rescaled = np.hstack([blue_rgb, np.ones((128, 1))])
    
    combined_colors = np.vstack([blue_rescaled, land_rescaled])
    combined = ListedColormap(combined_colors)
    
    vmin = np.nanmin(g.values)
    vmax = min(np.nanmax(g.values), 200) # print("Min:", vmin, "Max:", vmax)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    
    scatter_layer_g = LayerGroup()
    
    # step_elev = 7 # mit 50: 15 dots, mit 100: 8 dots
    for i in range(0, g.values.shape[0], step_elev.value):
        for j in range(0, g.values.shape[1], step_elev.value):
            val = g.values[i, j]
            if np.isnan(val):
                continue
            color = colors.to_hex(combined(norm(val)))
            marker = CircleMarker(
                location=(g.y.values[i], g.x.values[j]),
                radius=3,
                color=color,
                fill_color=color,
                fill_opacity=0.6
            )
            scatter_layer_g.add_layer(marker)
    
    # LEGEND ##############
    blue_colors = combined_colors[:128]
    land_colors = combined_colors[128:]
    def rounded_g(gmin = np.nanmin(g.values)):
        if gmin <= 500:
            return round(gmin / 50) * 50
        else:
            return round(gmin / 100) * 100
    vmin = rounded_g() # -50 # np.nanmin(g.values)
    vmax = min(np.nanmax(g), 200) # fz
    midpoint = 0
    blue_vals = np.linspace(vmin, midpoint, len(blue_colors), endpoint=False)
    land_vals = np.linspace(midpoint, vmax, len(land_colors))
    all_vals = np.concatenate([blue_vals, land_vals])
    all_colors = np.vstack([blue_colors, land_colors])
    all_colors_hex = [colors.to_hex(c) for c in all_colors]

    legend_elev = LinearColormap(
        colors=all_colors_hex,
        index=all_vals,
        vmin=vmin,
        vmax=vmax,
        caption="Elevation (m)"
    )

    # ADD TO MAP ################################################################
    # if scatter_layer_g in current_layer_group_SFINCS.value.layers:
    #     current_layer_group_SFINCS.value.remove_layer(scatter_layer_g)
    # current_layer_group_SFINCS.value.add_layer(scatter_layer_g)
    if elev_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(elev_layer.value)
    elev_layer.set(scatter_layer_g)
    current_layer_group_SFINCS.value.add_layer(elev_layer.value) # vorher erfolgreich, aber nicht zu resetten: current_layer_group_SFINCS.value.add_layer(scatter_layer_g)
    
    legend_html = widgets.HTML(value=legend_elev._repr_html_())
    legend_control = WidgetControl(widget=legend_html, position="bottomright")
    tab_controls_SFINCS.value["Elev"] = legend_control # m.add_control(legend_control)
    # control_update_signal_SFINCS.set(control_update_signal_SFINCS.value + 1)
    control_signals_SFINCS["Elev"].set(control_signals_SFINCS["Elev"].value + 1)

    plot_elev_status.set("Done")
    elev_plot_done_SFINCS.set(True)
    elevationdone_SFINCS.set(True) 



def reset_elev():
    if elev_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(elev_layer.value)

    if "Elev" in tab_controls_SFINCS.value:
        legend_control = tab_controls_SFINCS.value["Elev"]
        if legend_control in m_SFINCS.controls:
            m_SFINCS.remove_control(legend_control)
        tab_controls_SFINCS.value["Elev"] = None
    
    # Reset status or signals if needed
    control_signals_SFINCS["Elev"].set(control_signals_SFINCS["Elev"].value + 1)
    plot_elev_status.set("")
    elev_plot_done_SFINCS.set(False)
    step_elev.set(7)


In [9]:
@solara.component
def Tab_SFINCS_Elevation():
    solara.Markdown("**Setting step size for elevation plot:** Reducing the step size increases computational load and may significantly extend execution time.",style={"color": "inherit"})
    solara.SliderInt("Step Size", value=step_elev, min=-1, max=50)
    solara.Markdown(f"**Int value**: {step_elev.value}",style={"color": "inherit"})
    # with solara.Row():
    # solara.Button("Reset", on_click=lambda: step_elev.set(7)) 
    
    solara.Button(label="Plot elevation",on_click=plot_elev, continuous_update=True, style={"marginBottom": "20px"})
    if plot_elev_status.value == "Plotting":
        solara.Markdown("Preparing elevation data... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
    elif plot_elev_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})
    else:
        solara.Markdown("")

    solara.Button("Reset", on_click=reset_elev, disabled=not elev_plot_done_SFINCS.value, style={"marginBottom": "20px"})
    # if not elev_plot_done_SFINCS.value: 
    #     solara.Markdown("Plot elevation first.")    

    with solara.Row(justify="end"):
        solara.Button(label="Go to Step 4.", on_click=lambda: selected_tab_SFINCS.set('ActC'),disabled=not elevationdone_SFINCS.value)
    
# Tab_SFINCS_Elevation()


## 5) Active Cells

In [10]:
step_actc = solara.reactive(7)
actc_layer = solara.reactive(LayerGroup()) 
actc_plot_done_SFINCS = solara.reactive(False)
plot_actc_status = solara.reactive("Idle")

def plot_actc():
    global mask_active, msk, datasets_dep
    plot_actc_status.set("Plotting")


    # from tab grid generation
    sf.grid.create_from_region(                                  # UPDATE # sf.setup_grid_from_region()
        region={"bbox": ds_basins['geometry'].total_bounds},
        res=200,
        rotated=False,
        crs="utm"
    )
    datasets_dep = [{'elevtn': data_dict['topo'], 'zmin': 0.001}, {'elevtn': data_dict['bathy']}]
    _ = sf.elevation.create(elevation_list=datasets_dep) # UPDATE # _ = sf.setup_dep(datasets_dep=datasets_dep)
    
    # create mask
    mask_active = gpd.GeoDataFrame(geometry=ds_basins['geometry'])
    sf.mask.create_active(include_zmax=0, include_zmin=-30, include_polygon=mask_active, reset_mask=True) # UPDATE # sf.setup_mask_active(zmax=0, zmin=-30, include_mask=mask_active, reset_mask=True) 
    sf.mask.create_active(fill_area=10, reset_mask=False) # UPDATE # sf.setup_mask_active(fill_area=10, reset_mask=False)
    sf.mask.create_active(drop_area=10, reset_mask=False) # UPDATE # sf.setup_mask_active(drop_area=10, reset_mask=False)
    msk = sf.grid["msk"] # np.unique(msk.values) # print(np.count_nonzero(msk.values == 1)) # print(np.count_nonzero(msk.values == 0))
    
    
    # Transform locations from [m] to lat/lon
    transformer = Transformer.from_crs(msk.rio.crs , "EPSG:4326", always_xy=True)
    X, Y = np.meshgrid(msk.x.values, msk.y.values)
    lon, lat = transformer.transform(X, Y)
    
    
    # create scatter plot
    # # step_actc.value = 10 ######################## SET BY USER!
    # scatter_layer_m = LayerGroup()
    # for i in range(0, msk.values.shape[0], step_actc.value):
    #     for j in range(0, msk.values.shape[1], step_actc.value):
    #         val = msk.values[i, j]
    #         if np.isnan(val) or val == 0:
    #             continue
    #         if val == 1:
    #             color = "#666666"  # grey
    #         elif val == 2:
    #             color = "#e41a1c"  # red
    #         elif val == 3:
    #             color = "#984ea3"  # purple
    #         marker = CircleMarker(
    #             location=(lat[i, j], lon[i, j]), 
    #             radius=2,
    #             color=color,
    #             fill_color=color,
    #             fill_opacity=0.4
    #         )
    #         scatter_layer_m.add_layer(marker)

    
    scatter_layer_m = LayerGroup()    
    # First pass: plot grey dots with step
    for i in range(msk.shape[0]):
        for j in range(msk.shape[1]):
            val = msk.values[i, j]
            if val == 1 and i % step_actc.value == 0 and j % step_actc.value == 0:
                marker = CircleMarker(
                    location=(lat[i, j], lon[i, j]), 
                    radius=2,
                    color="#666666",  # grey
                    fill_color="#666666",
                    fill_opacity=0.4
                )
                scatter_layer_m.add_layer(marker)
    # Second pass: plot red and purple dots at full resolution
    for i in range(msk.shape[0]):
        for j in range(msk.shape[1]):
            val = msk.values[i, j]
            if val == 2:
                color = "#e41a1c"  # red
            elif val == 3:
                color = "#984ea3"  # purple
            else:
                continue
            marker = CircleMarker(
                location=(lat[i, j], lon[i, j]), 
                radius=2,
                color=color,
                fill_color=color,
                fill_opacity=0.8  # slightly more opaque to stand out
            )
            scatter_layer_m.add_layer(marker)    


    # ADD TO MAP ################################################################
    if actc_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(actc_layer.value)
    actc_layer.set(scatter_layer_m)
    current_layer_group_SFINCS.value.add_layer(actc_layer.value) # vorher erfolgreich, aber nicht zu resetten: current_layer_group_SFINCS.value.add_layer(scatter_layer_m)
    
    # Legend
    legend_html = widgets.HTML(value="""
    <div style="padding:5px; background:white; border-radius:5px; font-size:14px; color:black;">
        <b>msk</b><br>
        <div style="margin:2px;"><span style="display:inline-block;width:12px;height:12px;background:#666666;margin-right:5px;"></span> 1</div>
        <div style="margin:2px;"><span style="display:inline-block;width:12px;height:12px;background:#e41a1c;margin-right:5px;"></span> 2</div>
        <div style="margin:2px;"><span style="display:inline-block;width:12px;height:12px;background:#984ea3;margin-right:5px;"></span> 3</div>
    </div>
    """)
    legend_control = WidgetControl(widget=legend_html, position="bottomright")
    tab_controls_SFINCS.value["ActC"] = legend_control # m.add_control(legend_control)
    # control_update_signal_SFINCS.set(control_update_signal_SFINCS.value + 1)
    control_signals_SFINCS["ActC"].set(control_signals_SFINCS["ActC"].value + 1)


    plot_actc_status.set("Done")
    actc_plot_done_SFINCS.set(True)
    activecellsdone_SFINCS.set(True) 



def reset_actc():
    if actc_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(actc_layer.value)

    if "ActC" in tab_controls_SFINCS.value:
        legend_control = tab_controls_SFINCS.value["ActC"]
        if legend_control in m_SFINCS.controls:
            m_SFINCS.remove_control(legend_control)
        tab_controls_SFINCS.value["ActC"] = None
    
    # Reset status or signals if needed
    control_signals_SFINCS["ActC"].set(control_signals_SFINCS["ActC"].value + 1)
    plot_actc_status.set("")
    actc_plot_done_SFINCS.set(False)
    step_actc.set(7)


In [11]:
@solara.component
def Tab_SFINCS_Active_Cells():
    solara.Markdown("**Setting step size for plotting active cells:** Reducing the step size increases computational load and may significantly extend execution time.",style={"color": "inherit"})
    solara.SliderInt("Step Size", value=step_actc, min=-1, max=50)
    solara.Markdown(f"**Int value**: {step_actc.value}",style={"color": "inherit"})
    
    
    solara.Button(label="Plot active cells",on_click=plot_actc, continuous_update=True, style={"marginBottom": "20px"})
    if plot_actc_status.value == "Plotting":
        solara.Markdown("Preparing active cells data... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
    elif plot_actc_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})
    else:
        solara.Markdown("")


    # with solara.Row():
    solara.Button("Reset", on_click=reset_actc, disabled=not actc_plot_done_SFINCS.value, style={"marginBottom": "20px"})
    # if not actc_plot_done_SFINCS.value: 
    #     solara.Markdown("Plot active cells first.")

    with solara.Row(justify="end"):
        solara.Button(label="Go to Step 5.", on_click=lambda: selected_tab_SFINCS.set('wlBnd'),disabled=not activecellsdone_SFINCS.value)


    
# Tab_SFINCS_Active_Cells()

## 6) Waterlevel bound

In [12]:
step_wlbnd = solara.reactive(7)
wlbnd_layer = solara.reactive(LayerGroup()) 
wlbnd_plot_done_SFINCS = solara.reactive(False)
plot_wlbnd_status = solara.reactive("Idle")

def plot_wlbnd():
    global waterlevel_bnd
    plot_wlbnd_status.set("Plotting")

    # if mask_active does not exist:
    try:
        mask_active
    except NameError:
        mask_active = gpd.GeoDataFrame(geometry=ds_basins['geometry'])

    excl_bounds = gpd.GeoDataFrame(geometry=mask_active.to_crs(sf.crs).buffer(1000))
    sf.mask.create_boundary(btype='waterlevel', include_zmax=-5, reset_bounds=True, exclude_polygon=excl_bounds) # UPDATE # sf.setup_mask_bounds(btype='waterlevel', zmax=-5, reset_bounds=True, exclude_mask=excl_bounds)  # connectivity=4)
    excl_bounds_wgs84 = excl_bounds.to_crs(epsg=4326)
    geo_json_data = excl_bounds_wgs84.__geo_interface__


    try:
        msk
    except NameError:
        msk = sf.grid["msk"]
    
    mask_array = sf.grid.msk
    X, Y = np.meshgrid(mask_array.x.values, mask_array.y.values)
    transformer = Transformer.from_crs(msk.rio.crs , "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(X, Y)
    
    # step_wlbnd.value = 10 ###################### set by user
    scatter_layer_wlb = LayerGroup()    
    # First pass: plot grey dots with step
    for i in range(mask_array.shape[0]):
        for j in range(mask_array.shape[1]):
            val = mask_array.values[i, j]
            if val == 1 and i % step_wlbnd.value == 0 and j % step_wlbnd.value == 0:
                marker = CircleMarker(
                    location=(lat[i, j], lon[i, j]), 
                    radius=2,
                    color="#666666",  # grey
                    fill_color="#666666",
                    fill_opacity=0.4
                )
                scatter_layer_wlb.add_layer(marker)
    # Second pass: plot red and purple dots at full resolution
    waterlevel_bnd = [] # needed later for 7) River Inflow Points plot
    for i in range(mask_array.shape[0]):
        for j in range(mask_array.shape[1]):
            val = mask_array.values[i, j]
            if val == 2:
                color = "#e41a1c"  # red
            elif val == 3:
                color = "#984ea3"  # purple
            else:
                continue
    
            marker = CircleMarker(
                location=(lat[i, j], lon[i, j]), 
                radius=2,
                color=color,
                fill_color=color,
                fill_opacity=0.8  # slightly more opaque to stand out
            )
            scatter_layer_wlb.add_layer(marker)

            if val == 2:
                waterlevel_bnd.append(marker) # needed later for 7) River Inflow Points plot
    
    ########################################################################################################################

    # ADD TO MAP ################################################################
    if wlbnd_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(wlbnd_layer.value)
    wlbnd_layer.set(scatter_layer_wlb)
    current_layer_group_SFINCS.value.add_layer(wlbnd_layer.value) # vorher erfolgreich, aber nicht zu resetten: current_layer_group_SFINCS.value.add_layer(scatter_layer_wlb)

    
    # Legend
    legend_html = widgets.HTML(value="""
    <div style="padding:5px; background:white; border-radius:5px; font-size:14px; color:black;">
        <b>msk</b><br>
        <div style="margin:2px;"><span style="display:inline-block;width:12px;height:12px;background:#666666;margin-right:5px;"></span> 1</div>
        <div style="margin:2px;"><span style="display:inline-block;width:12px;height:12px;background:#e41a1c;margin-right:5px;"></span> 2</div>
        <div style="margin:2px;"><span style="display:inline-block;width:12px;height:12px;background:#984ea3;margin-right:5px;"></span> 3</div>
    </div>
    """)
    legend_control = WidgetControl(widget=legend_html, position="bottomright")
    tab_controls_SFINCS.value["wlBnd"] = legend_control # m.add_control(legend_control)
    # control_update_signal_SFINCS.set(control_update_signal_SFINCS.value + 1)
    control_signals_SFINCS["wlBnd"].set(control_signals_SFINCS["wlBnd"].value + 1)

    
    plot_wlbnd_status.set("Done")
    wlbnd_plot_done_SFINCS.set(True)
    waterlevelbnddone_SFINCS.set(True) 



def reset_wlbnd():
    if wlbnd_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(wlbnd_layer.value)

    if "wlBnd" in tab_controls_SFINCS.value:
        legend_control = tab_controls_SFINCS.value["wlBnd"]
        if legend_control in m_SFINCS.controls:
            m_SFINCS.remove_control(legend_control)
        tab_controls_SFINCS.value["wlBnd"] = None
    
    # Reset status or signals if needed
    control_signals_SFINCS["wlBnd"].set(control_signals_SFINCS["wlBnd"].value + 1)
    plot_wlbnd_status.set("")
    wlbnd_plot_done_SFINCS.set(False)
    step_wlbnd.set(7)




In [13]:
@solara.component
def Tab_SFINCS_Waterlevel_Bound():
    solara.Markdown("**Setting step size for plotting waterlevel bound:** Reducing the step size increases computational load and may significantly extend execution time.",style={"color": "inherit"})
    solara.SliderInt("Step Size", value=step_wlbnd, min=-1, max=50)
    solara.Markdown(f"**Int value**: {step_wlbnd.value}",style={"color": "inherit"})
    
    solara.Button(label="Plot waterlevel bound",on_click=plot_wlbnd, continuous_update=True, style={"marginBottom": "20px"})
    if plot_wlbnd_status.value == "Plotting":
        solara.Markdown("Preparing waterlevel bound data... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
    elif plot_wlbnd_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})
    else:
        solara.Markdown("")


    # with solara.Row():
    solara.Button("Reset", on_click=reset_wlbnd, disabled=not wlbnd_plot_done_SFINCS.value, style={"marginBottom": "20px"})
    # if not wlbnd_plot_done_SFINCS.value: 
    #     solara.Markdown("Plot waterlevel bound first.")

    with solara.Row(justify="end"):
        solara.Button(label="Go to Step 6.", on_click=lambda: selected_tab_SFINCS.set('rInflP'),disabled=not waterlevelbnddone_SFINCS.value)

    
# Tab_SFINCS_Waterlevel_Bound()

## 7) River Inflow Points

In [14]:
plot_rInflP_status = solara.reactive("Idle")
rInflP_layer = solara.reactive(LayerGroup())
rinflp_plot_done_SFINCS = solara.reactive(False)

def plot_rInflP():
    plot_rInflP_status.set("Plotting")
    global src_river_layer, riverinflow_layers

    # REMOVE existing layers if present
    if rInflP_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(rInflP_layer.value)
    if scatter_layer_g in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(scatter_layer_g)

    # Add Elevation Layer
    # rInflP_layer.set(scatter_layer_g)
    # current_layer_group_SFINCS.value.add_layer(rInflP_layer.value) 
    # vorher erfolgreich(er), aber nicht zu resetten: 
    current_layer_group_SFINCS.value.add_layer(scatter_layer_g)

    combined_layer = LayerGroup()
    src_river_layer = LayerGroup()
    
    # Add RIVERS
    sf.river.create_inflow(                                  # UPDATE # sf.setup_river_inflow(
        hydrography=data_dict["hydrography"],
        river_len=1000,
        river_upa=10,
        keep_rivers_geom=True
    )

    rivers_inflow = sf.geoms["rivers_inflow"]
    transformer = Transformer.from_crs(rivers_inflow.crs, "EPSG:4326", always_xy=True)

    for geom in rivers_inflow["geometry"]:
        if geom.geom_type == "LineString":
            x, y = geom.xy
            lon, lat = transformer.transform(x, y)
            locations = list(zip(lat, lon))
            polyline = Polyline(locations=locations, color="blue", weight=2, fill=False)
            combined_layer.add_layer(polyline)
            src_river_layer.add_layer(polyline)

    # Add SOURCE POINTS
    gdf_pnt = river_source_points(
        gdf_riv=rivers_inflow,
        gdf_mask=mask_active,
        src_type='inflow'
    )
    gdf_pnt = gdf_pnt.to_crs(rivers_inflow.crs)
    transformer = Transformer.from_crs(gdf_pnt.crs, "EPSG:4326", always_xy=True)

    text_step = 0.035
    label_markers = []
    
    for idx, geom in enumerate(gdf_pnt.geometry):
        if geom.geom_type == "Point":
            lon, lat = transformer.transform(geom.x, geom.y)
            marker = CircleMarker(
                location=(lat, lon),
                radius=5,
                color="black",
                weight=1,
                fill=True,
                fill_color="white",
                fill_opacity=1.0
            )
            combined_layer.add_layer(marker)
            src_river_layer.add_layer(marker)


###################################################### THIS PART!!! ######################################################
            label = Marker(
                location=(lat + text_step, lon),  # Slight offset to avoid overlap
                icon=Icon(
                    icon_url="data:image/svg+xml;charset=utf-8," + 
                    f"<svg xmlns='http://www.w3.org/2000/svg' width='30' height='20'><text x='0' y='15' font-size='20' font-weight='bold' fill='black'>{idx+1}</text></svg>",
                    icon_size=[30, 20]
                )
            )
            # mmm.add_layer(label)
            combined_layer.add_layer(label)
            src_river_layer.add_layer(label)
            label_markers.append((label, lat, lon)) 

    def update_text_step(change):
        global text_step
        zoom = change['new']
        if zoom <= 7:
            text_step = 0.035
        elif zoom == 8:
            text_step = 0.025
        elif zoom == 9:
            text_step = 0.015
        elif zoom == 10:
            text_step = 0.01
        else:
            text_step = 0.01 * (0.5 ** (zoom-10))
    
        for label, lat, lon in label_markers:
            label.location = (lat + text_step, lon)

    m_SFINCS.observe(update_text_step, names='zoom')

############################################################################################################


    
    # Add WATERLEVEL BOUND markers
    for marker in waterlevel_bnd:
        combined_layer.add_layer(marker)

    # Update layer on map
    if rInflP_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(rInflP_layer.value)
    rInflP_layer.set(combined_layer)
    current_layer_group_SFINCS.value.add_layer(rInflP_layer.value) # vorher erfolgreich, aber nicht zu resetten: current_layer_group_SFINCS.value.add_layer(combined_layer)

    # Legend: Elevation
    legend_elev_html = widgets.HTML(value=legend_elev._repr_html_())
    legend_elev_control = WidgetControl(widget=legend_elev_html, position="bottomright")
    
    # Create legend for rivers + sources + WLB
    legend_custom_html = """
    <div style="padding: 10px; background: white; color: black; border: 1px solid gray; border-radius: 5px; font-size: 13px;">
        <b>Legend</b><br>
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 12px; background-color: red; border-radius: 50%; margin-right: 5px;"></div>
            Waterlevel bound
        </div>
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 3px; background-color: blue; margin-right: 5px;"></div>
            Rivers inflow
        </div>
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 12px; background-color: white; border: 1px solid black; border-radius: 50%; margin-right: 5px;"></div>
            Sources
        </div>
    </div>
    """
    legend_widget = widgets.HTML(value=legend_custom_html)
    legend_control = WidgetControl(widget=legend_widget, position="bottomleft")

    tab_controls_SFINCS.value["rInflP"] = legend_control
    control_signals_SFINCS["rInflP"].set(control_signals_SFINCS["rInflP"].value + 1)

    plot_rInflP_status.set("Done")
    rinflp_plot_done_SFINCS.set(True)
    riverinflowdone_SFINCS.set(True) 

    riverinflow_layers = LayerGroup(layers=list(current_layer_group_SFINCS.value.layers)) # HIER HIER HIER HIER HIER HIER HIER HIER HIER HIER
    # tab_controls_SFINCS.value["rInflP"] and control_signals_SFINCS["rInflP"]
    


def reset_rInflP():
    reset_elev()
    if rInflP_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(rInflP_layer.value)

    if scatter_layer_g in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(scatter_layer_g)

    if "rInflP" in tab_controls_SFINCS.value:
        legend_control = tab_controls_SFINCS.value["rInflP"]
        if legend_control in m_SFINCS.controls:
            m_SFINCS.remove_control(legend_control)
        tab_controls_SFINCS.value["rInflP"] = None
    
    # Reset status or signals if needed
    plot_rInflP_status.set("")
    control_signals_SFINCS["rInflP"].set(control_signals_SFINCS["rInflP"].value + 1)
    rinflp_plot_done_SFINCS.set(False)


In [15]:
@solara.component
def Tab_SFINCS_River_Inflow_Points():
    solara.Markdown("**Show River Inflow Points**",style={"color": "inherit"})

    solara.Markdown("**Note**: the underlying elevation data comes from Tab 3. ELEVATION",style={"color": "inherit"})
        
    solara.Button(label="Plot river inflow points",on_click=plot_rInflP, continuous_update=True, style={"marginBottom": "20px"})
    
    
    if plot_rInflP_status.value == "Plotting":
        solara.Markdown("Preparing river inflow sources... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
    elif plot_rInflP_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})
    else:
        solara.Markdown("")


    solara.Button("Reset", on_click=reset_rInflP, disabled=not rinflp_plot_done_SFINCS.value, style={"marginBottom": "20px"})
    # if not rinflp_plot_done_SFINCS.value: 
    #     solara.Markdown("Plot river inflow points first.")

    with solara.Row(justify="end"):
        solara.Button(label="Go to Step 7.", on_click=lambda: selected_tab_SFINCS.set('subgrid'),disabled=not riverinflowdone_SFINCS.value)

    
# Tab_SFINCS_River_Inflow_Points()

## 8) Land roughness and infiltration

Implement rather in 9) Subgrid

In [16]:
LRI_status = solara.reactive("Idle")

def LRI_get_data():
    global datasets_rgh
    LRI_status.set("Running")
    sf.infiltration.create_cn(data_dict['infiltration'], antecedent_moisture='avg') # Setup soil infiltration # UPDATE # sf.setup_cn_infiltration(data_dict['infiltration'], antecedent_moisture='avg')
    datasets_rgh = [{'lulc': data_dict['lulc']}] # Setup surface roughness
    # sf.grid.data_vars.keys() # check all variables in the sf.grid dataset
    info_retrieved_done_SFINCS.set(True)
    LRI_status.set("Done")
    

In [17]:
# @solara.component
# def Tab_SFINCS_Land_Roughness_Infiltration():
#     solara.Markdown("**Get information about land roughness and soil infiltration**")
    
    
#     solara.Button(label="Retrieve information",on_click=LRI_get_data, continuous_update=True)
#     if LRI_status.value == "Running":
#         solara.Markdown("Retrieving information... Please wait.")
#     elif LRI_status.value == "Done":
#         solara.Markdown("**Done!**")


# # Tab_SFINCS_Land_Roughness_Infiltration()

## 9) Subgrid

rivers & source points bleiben; kann ich später hinzu fügen. jetzt brauche ich z_zmin

NUMBER OF SUBGRID_PIXELS SHOULD BE SET BY USER (with a default value)!!! because: if (in this example) e.g. grid resolution = 200 m, DEM resolution is around 1 arcsec ~ 30m --> 200/30 --> nr_subgrid_pixels = 6

In [18]:
zzmin_status = solara.reactive("Idle")
subgrid_layer = solara.reactive(LayerGroup())
subgrid_plot_done_SFINCS = solara.reactive(False)
info_retrieved_done_SFINCS = solara.reactive(False)
nr_subgrid_pixels = solara.reactive(6)

def plot_zzmin():
    global scatter_layer_sg
    zzmin_status.set("Running")

    if subgrid_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(subgrid_layer.value)

    sf.subgrid.create(                               # UPDATE # sf.setup_subgrid(
        elevation_list=datasets_dep,  # The elevation data
        roughness_list=datasets_rgh,  # The roughness data
        nr_subgrid_pixels=nr_subgrid_pixels.value,        # TO BE SET BY USER!!!!!
        write_dep_tif=True,         # Whether to write the subgrid elevation data
        write_man_tif=False         # Whether to write the subgrid roughness data
    )

    step_sg = solara.reactive(7)
    zzmin = sf.subgrid.z_zmin
    
    transformer = Transformer.from_crs(zzmin.rio.crs , "EPSG:4326", always_xy=True)
    X, Y = np.meshgrid(zzmin.x.values, zzmin.y.values)
    lon, lat = transformer.transform(X, Y)
    
    colormap = LinearColormap(
            colors=["blue", "white", "red"],
            vmin=-float(np.max(zzmin)), # float(np.min(zzmin)), 
            vmax=float(np.max(zzmin)),
            caption="z_zmin"
        )
    
    step_sg.value = 10 #########################################
    scatter_layer_sg = LayerGroup()
    for i in range(0, zzmin.values.shape[0], step_sg.value):
        for j in range(0, zzmin.values.shape[1], step_sg.value):
            val = zzmin.values[i, j]
            if np.isnan(val):
                continue
            color = colormap(val)
    
            marker = CircleMarker(
                location=(lat[i, j], lon[i, j]), 
                radius=2,
                color=color,
                fill_color=color,
                fill_opacity=0.4
            )
            scatter_layer_sg.add_layer(marker)
    
    # LEGEND ##############
    vmin = np.round(-float(np.max(zzmin)))
    vmax = np.round(float(np.max(zzmin)))
    midpoint = 0

    legend_zzmin = LinearColormap(
        colors=["blue", "white", "red"],
        # index=all_vals,
        vmin=vmin,
        vmax=vmax,
        caption="z_zmin"
    )

    # ADD TO MAP ################################################################
    if subgrid_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(subgrid_layer.value)
    subgrid_layer.set(scatter_layer_sg)
    # current_layer_group_SFINCS.value.add_layer(subgrid_layer.value) # vorher erfolgreich, aber nicht zu resetten: current_layer_group_SFINCS.value.add_layer(scatter_layer_sg)
    current_layer_group_SFINCS.value.add_layer(scatter_layer_sg)


    
    if src_river_layer in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(src_river_layer)
    # # subgrid_layer.set(src_river_layer) # das hier war raus
    # current_layer_group_SFINCS.value.add_layer(subgrid_layer.value) # vorher erfolgreich, aber nicht zu resetten: current_layer_group_SFINCS.value.add_layer(src_river_layer)
    current_layer_group_SFINCS.value.add_layer(src_river_layer)

    legend_html = widgets.HTML(value=legend_zzmin._repr_html_())
    legend_control = WidgetControl(widget=legend_html, position="bottomright")
    tab_controls_SFINCS.value["subgrid"] = legend_control
    control_signals_SFINCS["subgrid"].set(control_signals_SFINCS["subgrid"].value + 1)
    
    zzmin_status.set("Done")
    subgrid_plot_done_SFINCS.set(True)
    roughnesssubgriddone_SFINCS.set(True) 


def reset_subgrid():
    if subgrid_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(subgrid_layer.value)

    if scatter_layer_sg in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(scatter_layer_sg)
        
    if src_river_layer in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(src_river_layer)

    if "subgrid" in tab_controls_SFINCS.value:
        legend_control = tab_controls_SFINCS.value["subgrid"]
        if legend_control in m_SFINCS.controls:
            m_SFINCS.remove_control(legend_control)
        tab_controls_SFINCS.value["subgrid"] = None
    
    # Reset status or signals if needed
    control_signals_SFINCS["subgrid"].set(control_signals_SFINCS["subgrid"].value + 1)
    zzmin_status.set("")
    nr_subgrid_pixels.set(6)
    subgrid_plot_done_SFINCS.set(False)



In [19]:

@solara.component
def Tab_SFINCS_Subgrid():
    solara.Markdown("**Get information about land roughness and soil infiltration**",style={"color": "inherit"})
    solara.Button(label="Retrieve information",on_click=LRI_get_data, style={"marginBottom": "20px"})#, continuous_update=True)
    if LRI_status.value == "Running":
        solara.Markdown("Retrieving information... Please wait.",style={"color": "inherit"})
    elif LRI_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})

    ###############################
    

    solara.Markdown("**Setting the number of subgrid pixels:** Ratio of grid resolution to DEM resolution.",style={"color": "inherit"})
    solara.InputInt("Number of subgrid pixels", value=nr_subgrid_pixels, style={"marginTop": "20px"})#, continuous_update=True)
    solara.Markdown(f"**Selected**: {nr_subgrid_pixels.value}",style={"color": "inherit"})
    # with solara.Row():
    solara.Button("Set to Default", on_click=lambda: nr_subgrid_pixels.set(6), style={"marginBottom": "20px"})
    
    solara.Button(label="Plot",on_click=plot_zzmin, disabled=not info_retrieved_done_SFINCS.value, style={"marginBottom": "20px"})#, continuous_update=True)
    if not info_retrieved_done_SFINCS.value: 
        solara.Markdown("Retrieve information first.",style={"color": "inherit"})
    if zzmin_status.value == "Running":
        solara.Markdown("Plot zzmin... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
    elif zzmin_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})
    else:
        solara.Markdown("")

    solara.Button("Reset", on_click=reset_subgrid, disabled=not subgrid_plot_done_SFINCS.value, style={"marginBottom": "20px"})
    # if not subgrid_plot_done_SFINCS.value: 
    #     solara.Markdown("Plot subgrid first.")

    with solara.Row(justify="end"):
        solara.Button(label="Go to Step 8.", on_click=lambda: selected_tab_SFINCS.set('obs'),disabled=not roughnesssubgriddone_SFINCS.value)

    
# Tab_SFINCS_Subgrid()

## 10) Obs stations

In [20]:
# roughnesssubgriddone_SFINCS.set(True) 
obs_layer = solara.reactive(LayerGroup())
obs_points_fn = solara.reactive("P:\\11210647-iriscc\\Modelbuilder_SFINCS\\data\\sfincs_Humber_obs.geojson")# ("C:\\") # HIER PATH ÄNDERN

def empty_gdf():
    return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

def show_imported_ob_loc():
    global obs
    sf.geoms.pop("obs", None) 
    
    sf.observation_points.create(locations=obs_points_fn.value) # UPDATE # sf.setup_observation_points(locations=obs_points_fn.value)
    obs = sf.geoms["obs"]

    transformer = Transformer.from_crs(obs.crs, "EPSG:4326", always_xy=True)
    
    imp_obs_layer = LayerGroup()
    icon_bino = AwesomeIcon(name='binoculars',marker_color='red',icon_color='black',spin=False)
    
    for idx, geom in enumerate(obs.geometry):
        if geom.geom_type == "Point":
            lon, lat = transformer.transform(geom.x, geom.y)
            marker = Marker(icon=icon_bino, location=(lat, lon), draggable=False)
            imp_obs_layer.add_layer(marker)
    # mmm.add_layer(imp_obs_layer) # ADJUST TO STRUCTURE OF LAYERS!!! 
    
    # ADD TO MAP ################################################################
    if obs_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(obs_layer.value)
    obs_layer.set(imp_obs_layer)
    current_layer_group_SFINCS.value.add_layer(obs_layer.value) # .add_layer(scatter_layer_sg)


def remove_imported_ob_loc():
    global obs, all_obs_loc
    if obs_layer.value in current_layer_group_SFINCS.value.layers:
        current_layer_group_SFINCS.value.remove_layer(obs_layer.value)
    obs_layer.set(LayerGroup())
    obs = None
    # sf.geoms["obs"] = empty_gdf() # [] # sf.geoms["obs"] = None # sf.geoms.pop("obs", None) 
    sf.geoms.clear() # sf.geoms.pop("obs", None); 
    all_obs_loc = empty_gdf()



place_obs_enabled = solara.reactive(False)
locations = []

def handle_map_click(**kwargs):
    # place_obs.set("activated")
    if not place_obs_enabled.value:
        return
    if kwargs.get("type") == "click":
        latlng = kwargs.get("coordinates")
        user_obs_loc = Marker(location=latlng, draggable=False)
        m_SFINCS.add_layer(user_obs_loc)
        locations.append(latlng)

        # remove marker on click
        def remove_marker(**_):
            if user_obs_loc in m_SFINCS.layers:
                m_SFINCS.remove_layer(user_obs_loc)
            if user_obs_loc.location in locations:
                locations.remove(user_obs_loc.location)
        user_obs_loc.on_click(remove_marker)

m_SFINCS.on_interaction(handle_map_click)


save_obs_status = solara.reactive("Idle")
def fix_obs_loc():
    global all_obs_loc
    save_obs_status.set("Running")
    try:
        obs_exists = "obs" in globals() and obs is not None and not obs.empty
    except Exception:
        obs_exists = False

    loc_exists = "locations" in globals() and len(locations) > 0
    gdfs = []

    if obs_exists:

        obs_wgs = obs.to_crs("EPSG:4326")
        gdfs.append(obs_wgs)
        obs_wgs = obs

    if loc_exists:
        # locations are already lat/lon → build GeoDataFrame in EPSG:4326
        user_geoms = [Point(lon, lat) for lat, lon in locations]
        user_obs = gpd.GeoDataFrame(geometry=user_geoms, crs="EPSG:4326")
        gdfs.append(user_obs)

    if gdfs:
        print("building all_obs_loc")
        all_obs_loc = gpd.GeoDataFrame(
            pd.concat(gdfs, ignore_index=True), crs="EPSG:4326"
        )
        sf.geoms["obs"] = all_obs_loc
    else:
        print("no obs, no locations -> empty")
        all_obs_loc = empty_gdf()
        sf.geoms["obs"] = empty_gdf()

    # disable interactions again
    m_SFINCS.interaction_callbacks = []
    save_obs_status.set("Done")
    obsdone_SFINCS.set(True) 



In [21]:
@solara.component
def Tab_SFINCS_Observations():
    solara.Markdown("**Observation Locations:** Import existing observation locations and add new ones on the map.",style={"color": "inherit"})
    solara.Markdown("**Select Model:**",style={"color": "inherit"})
    solara.InputText("Model directory", value=obs_points_fn, continuous_update=True, style={"marginBottom": "20px"})
    path_imported_obs = Path(obs_points_fn.value)
    if obs_points_fn.value:
        if not obs_points_fn.value.endswith(".geojson"):
            solara.Error(label='Error: File must end with ".geojson"', text=False, dense=True, outlined=True, icon=False)
        elif not path_imported_obs.exists():
            solara.Error("File does not exist.")

    with solara.Row(gap="25px"): 
        solara.Button(label="Import & Show",on_click=show_imported_ob_loc)
        solara.Button(label="Remove",on_click=remove_imported_ob_loc)

    solara.Markdown("Imported observation locations are shown in red.",style={"color": "inherit"})

    
        # solara.Button(label="Place additional observation locations on map",on_click=enable_placing_locs)
    solara.Switch(label="Enable placing markers (in blue)", value=place_obs_enabled)
    # solara.use_effect(lambda: (locations.clear(),m.interaction_callbacks.clear(),m.on_interaction(handle_map_click) if place_obs_enabled.value else None))
    
    if place_obs_enabled.value == "activated": 
        solara.Markdown("Click on map to place marker. Click on marker to remove.",style={"color": "inherit"})
    else:
        solara.Markdown("")
    
    solara.Button(label="Save all locations", on_click=fix_obs_loc)
    if save_obs_status.value == "Running":
            solara.Markdown("Saving Observations... Please wait.",style={"color": "inherit"})
    elif save_obs_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})

    with solara.Row(justify="end"):
        solara.Button(label="Go to Step 9.", on_click=lambda: selected_tab_SFINCS.set('forcing'),disabled=not obsdone_SFINCS.value)

    
# Tab_SFINCS_Observations()

## 11) Set-up forcings

In [22]:
wl_distance = solara.reactive(10000)
forcing_source_options = ["era5_hourly", "Other (not yet implemented)"]
forcing_source = solara.reactive("era5_hourly")

wl_bnd_forcing_status = solara.reactive("Idle")
def setup_wl_bnd():
    wl_bnd_forcing_status.set("Running")
    sf.water_level.create_boundary_points_from_mask(bnd_dist=wl_distance.value,merge=True) # UPDATE # sf.setup_waterlevel_bnd_from_mask(distance=wl_distance.value,merge=True)
    wl_bnd_forcing_status.set("Done")

forcings_status = solara.reactive("Idle")
def setup_meteo():
    forcings_status.set("Running")
    sf.precipitation.create(precip=forcing_source.value, aggregate=False) # UPDATE # sf.setup_precip_forcing_from_grid(precip=forcing_source.value, aggregate=False)
    sf.pressure.create(press=forcing_source.value) # UPDATE # sf.setup_pressure_forcing_from_grid(press=forcing_source.value)
    sf.wind.create(wind=forcing_source.value) # UPDATE # sf.setup_wind_forcing_from_grid(wind=forcing_source.value) 
    forcings_status.set("Done")
    forcingdone_SFINCS.set(True)



In [23]:
@solara.component
def Tab_SFINCS_setup_forcings():
    solara.Markdown("**Setup waterlevel boundaries**",style={"color": "inherit"})
    solara.InputInt("Number of subgrid pixels", value=wl_distance, style={"marginBottom": "20px"})
    solara.Button("Setup waterlevel boundaries",on_click=setup_wl_bnd, style={"marginBottom": "20px"})

    if wl_bnd_forcing_status.value == "Running":
        solara.Markdown("Setting-up waterlevel boundaries... Please wait.",style={"color": "inherit"})
    elif wl_bnd_forcing_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})        

    solara.Markdown("**Setup spatially varying meteo data**",style={"color": "inherit"})
    solara.Select(label="forcing??? can this vary??", value=forcing_source, values=forcing_source_options) # solara.InputText("forcing???", value=forcing_source, continuous_update=True)
    solara.Button("Setup",on_click=setup_meteo, style={"marginBottom": "20px"})

    if forcings_status.value == "Running":
        solara.Markdown("Setting-up spatially varying meteo data... Please wait.",style={"color": "inherit"})
    elif forcings_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})


    with solara.Row(justify="end"):
        solara.Button(label="Go to Step 10.", on_click=lambda: selected_tab_SFINCS.set('model'),disabled=not forcingdone_SFINCS.value)
        

# Tab_SFINCS_setup_forcings()

## 12) Show and Write Model

In [24]:
def show_wlbnd():
    bnd_df = pd.read_csv(os.path.join(model_path.value, 'sfincs.bnd'), delim_whitespace=True, header=None, names=['x', 'y'])
    transformer = Transformer.from_crs(obs.crs, "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(bnd_df['x'].values, bnd_df['y'].values)
    
    wlbnd_point_layer = LayerGroup()
    text_step = 0.035
    label_markers_wlbnd = []
    
    for idx, (lon_i, lat_i) in enumerate(zip(lon, lat)):# for lon, lat in list(zip(lon, lat)):
        marker_wl = CircleMarker(
            location=(lat_i, lon_i),  # Note: folium and ipyleaflet use (lat, lon)
            radius=5,
            color="black",
            weight=1,
            fill=True,
            fill_color="grey",
            fill_opacity=1.0
        )
        wlbnd_point_layer.add_layer(marker_wl)
    
        # text_step = 0.035
        # label_markers_wlbnd = []
    
        label_wl = Marker(
            location=(lat_i + text_step, lon_i),  # Slight offset to avoid overlap
            icon=Icon(
                icon_url="data:image/svg+xml;charset=utf-8," + 
                f"<svg xmlns='http://www.w3.org/2000/svg' width='30' height='20'><text x='0' y='15' font-size='20' font-weight='bold' fill='black'>{idx+1}</text></svg>",
                icon_size=[30, 20]
            )
        )
        # mmm.add_layer(label)
        wlbnd_point_layer.add_layer(label_wl)
        label_markers_wlbnd.append((label_wl, lat_i, lon_i))
    
    def update_text_step(change):
        global text_step
        zoom = change['new']
        if zoom <= 7:
            text_step = 0.035
        elif zoom == 8:
            text_step = 0.025
        elif zoom == 9:
            text_step = 0.015
        elif zoom == 10:
            text_step = 0.01
        else:
            text_step = 0.01 * (0.5 ** (zoom-10))
        
        for label, lat, lon in label_markers:
            label.location = (lat + text_step, lon)
    
    m_SFINCS.observe(update_text_step, names='zoom')

    # if rInflP_layer.value in current_layer_group_SFINCS.value.layers:
    #     current_layer_group_SFINCS.value.remove_layer(rInflP_layer.value)
    # rInflP_layer.set(wlbnd_point_layer)
    # current_layer_group_SFINCS.value.add_layer(rInflP_layer.value)


    # Create legend for rivers + sources + WLB
    legend_custom_html_model = """
    <div style="padding: 10px; background: white; color: black; border: 1px solid gray; border-radius: 5px; font-size: 13px;">
        <b>Legend</b><br>
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 12px; background-color: red; border-radius: 50%; margin-right: 5px;"></div>
            Waterlevel bound
        </div>
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 3px; background-color: blue; margin-right: 5px;"></div>
            Rivers inflow
        </div>
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 12px; background-color: white; border: 1px solid black; border-radius: 50%; margin-right: 5px;"></div>
            Sources
        </div>
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 12px; background-color: grey; border: 1px solid black; border-radius: 50%; margin-right: 5px;"></div>
            Waterlevel Bounds
        </div>
    </div>
    """
    legend_widget_model = widgets.HTML(value=legend_custom_html_model)
    legend_control_model = WidgetControl(widget=legend_widget_model, position="bottomleft")

    tab_controls_SFINCS.value["model"] = legend_control_model


    return wlbnd_point_layer

    

In [25]:
write_model_status = solara.reactive("Idle")
def write_model():
    write_model_status.set("Running")

    sf.write()
    sf_new = SfincsModel(root=sf_root.value, mode='r')
    sf_new.read()

    write_model_status.set("Done")

show_tree = solara.reactive(False)
tree_output = solara.reactive("")
def tree(directory):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
    
        print(f"+ {directory}")
        for path in sorted(directory.rglob('*')):
            depth = len(path.relative_to(directory).parts)
            spacer = "  " * depth
            print(f"{spacer}+ {path.name}")

    return buffer.getvalue()

##############################################################################################################################################

def figure_dis():
    loading_dis.value = True
    def worker():
        f, ax = plt.subplots(figsize=(8,5), constrained_layout=True, dpi = 100)
        for i in range(sf.forcing['dis'].shape[1]):  # 15 indices
            ax.plot(sf.forcing['dis'].time, sf.forcing['dis'].values[:, i], label=f"{i+1}")
        ax.set_xlabel("Time"); ax.set_ylabel("Discharge [m³/s]")
        ax.legend(title="Index", bbox_to_anchor=(1.05, 1), loc="upper left")
        # f.tight_layout()
        fig_dis.value = f
        loading_dis.value = False
        fig_ready_dis.value = True
    threading.Thread(target=worker).start()

def reset_figure_dis():
    fig_dis.value = None
    fig_ready_dis.value = False
    loading_dis.value = False


In [26]:

show_dis = solara.reactive(False); loading_dis = solara.reactive(False); fig_ready_dis = solara.reactive(False); fig_dis = solara.reactive(None)
show_bzs = solara.reactive(False)
show_precip2d = solara.reactive(False)
show_press2d = solara.reactive(False)
show_wind10u = solara.reactive(False)
show_wind10v = solara.reactive(False)


@solara.component
def Tab_SFINCS_show_write_model():
    solara.Markdown("**Show and Write Model:**",style={"color": "inherit"})

    solara.Button("Write Model",on_click=write_model)
    if write_model_status.value == "Running":
        solara.Markdown("**Running ...** Please wait.",style={"color": "inherit"})
    elif write_model_status.value == "Done":
        solara.Markdown("**Model set-up successfully!**",style={"color": "inherit"})

    solara.Switch(label="Show Path Structure", value=show_tree, disabled=(write_model_status.value != "Done"))
    if show_tree.value: # tree(Path(sf.root))
        tree_output.value = tree(Path(sf.root))
        solara.Markdown(f"```\n{tree_output.value}\n```",style={"color": "inherit"})

    solara.Markdown("**Show Forcings**, but note: figures might take some seconds to appear.",style={"color": "inherit"})
    
    solara.Switch(label="Show SFINCS discharge forcing", value=show_dis, disabled=(write_model_status.value != "Done")) #########################################################
    # if show_dis.value: # plt.plot(sf.forcing['dis'].time,sf.forcing['dis'].values)
    #     fig, ax = plt.subplots(figsize=(8, 5))        
    #     for i in range(sf.forcing['dis'].shape[1]):  # 15 indices
    #         ax.plot(sf.forcing['dis'].time, sf.forcing['dis'].values[:, i], label=f"{i+1}")
    #     ax.set_xlabel("Time"); ax.set_ylabel("Discharge [m³/s]")
    #     ax.legend(title="Index", bbox_to_anchor=(1.05, 1), loc="upper left")
    #     fig.tight_layout()
    #     # show_dis_status.set("Done")
    #     solara.FigureMatplotlib(fig)
    if loading_dis.value:
        solara.Markdown("⏳ Generating figure...",style={"color": "inherit"})
    if fig_ready_dis.value and fig_dis.value is not None:
        solara.FigureMatplotlib(fig_dis.value)
    solara.use_effect(
    lambda: figure_dis() if show_dis.value and not fig_ready_dis.value else reset_figure_dis(),
    dependencies=[show_dis.value])


    
        

    solara.Switch(label="Show SFINCS waterlevel forcing", value=show_bzs, disabled=(write_model_status.value != "Done"))
    if show_bzs.value: # plt.plot(sf.forcing['bzs'].time,sf.forcing['bzs'].values)
        fig, ax = plt.subplots(figsize=(8, 5))        
        for i in range(sf.forcing['bzs'].shape[1]):  # 15 indices
            ax.plot(sf.forcing['bzs'].time, sf.forcing['bzs'].values[:, i], label=f"{i}")
        ax.set_xlabel("Time"); ax.set_ylabel("Waterlevel [m+ref]")
        ax.legend(title="Index", bbox_to_anchor=(1.05, 1), loc="upper left")
        fig.tight_layout()
        solara.FigureMatplotlib(fig)

    solara.Switch(label="Show SFINCS precipitation forcing", value=show_precip2d, disabled=(write_model_status.value != "Done"))
    if show_precip2d.value:
        fig, ax = plt.subplots(figsize=(8, 5))    
        da = sf.forcing['precip_2d'].transpose("time", ...)
        da = da.mean(dim=[da.raster.x_dim, da.raster.y_dim])
        df = da.to_pandas()
        if isinstance(df.index, pd.MultiIndex):
            df = df.unstack(0)
        df.index = mdates.date2num(df.index)
        ax.bar(df.index, df.values, facecolor="darkblue"); del df, da
        ax.set_xlabel("Time"); ax.set_ylabel("Mean Precipitation [mm/hr]")
        fig.tight_layout()
        solara.FigureMatplotlib(fig)

    solara.Switch(label="Show SFINCS barometric pressure forcing", value=show_press2d, disabled=(write_model_status.value != "Done"))
    if show_press2d.value:
        fig, ax = plt.subplots(figsize=(8, 5))    
        da = sf.forcing['press_2d'].transpose("time", ...)
        da = da.mean(dim=[da.raster.x_dim, da.raster.y_dim])
        df = da.to_pandas()
        if isinstance(df.index, pd.MultiIndex):
            df = df.unstack(0)
        df.index = mdates.date2num(df.index)
        ax.plot(df); del df, da
        ax.set_xlabel("Time"); ax.set_ylabel("Min barometric Pressure [Pa]")
        fig.tight_layout()
        solara.FigureMatplotlib(fig)
  
    solara.Switch(label="Show SFINCS eastward wind forcing", value=show_wind10u, disabled=(write_model_status.value != "Done"))
    if show_wind10u.value:
        fig, ax = plt.subplots(figsize=(8, 5))  
        da = sf.forcing['wind10_u'].transpose("time", ...)
        if da.ndim == 3:
            da = da.mean(dim=[da.raster.x_dim, da.raster.y_dim])
        df = da.to_pandas()
        ax.plot(df); del df, da
        ax.set_xlabel("Time"); ax.set_ylabel("Mean eastward Wind [m/s]")
        fig.tight_layout()
        solara.FigureMatplotlib(fig)
  
    solara.Switch(label="Show SFINCS northward wind forcing", value=show_wind10v, disabled=(write_model_status.value != "Done"))
    if show_wind10v.value:
        fig, ax = plt.subplots(figsize=(8, 5))  
        da = sf.forcing['wind10_v'].transpose("time", ...)
        if da.ndim == 3:
            da = da.mean(dim=[da.raster.x_dim, da.raster.y_dim])
        df = da.to_pandas()
        ax.plot(df); del df, da
        ax.set_xlabel("Time"); ax.set_ylabel("Mean northwards Wind [m/s]")
        fig.tight_layout()
        solara.FigureMatplotlib(fig)

# Tab_SFINCS_show_write_model()

# D-HYDRO

## 1) Import & Settings

In [27]:
TAB_NAMES_DHYDRO = ["UserInput", "GridGen","Bnd","Forcing","Obs","mdu","run","visu1","visu2","visu3"]
tab_layers_DHYDRO = {name: LayerGroup() for name in TAB_NAMES_DHYDRO}

selected_tab_DHYDRO = solara.reactive('UserInput') 

current_layer_group_DHYDRO = solara.reactive(tab_layers_DHYDRO['UserInput'])

tab_controls_DHYDRO = solara.reactive({})
control_update_signal_DHYDRO = solara.reactive(0)
current_control_DHYDRO = solara.reactive(None)

userinputdone_DHYDRO = solara.reactive(False) 
gridgendone_DHYDRO = solara.reactive(False); grid_generated_DHYDRO = solara.reactive(False); grid_refined_DHYDRO = solara.reactive(False)
bnddone_DHYDRO = solara.reactive(False); tidalbnddone_DHYDRO = solara.reactive(False); downloadGTSMdone_DHYDRO = solara.reactive(False)
forcingdone_DHYDRO = solara.reactive(False); genextforcingdone_DHYDRO = solara.reactive(False)#; downloadERA5done_DHYDRO = solara.reactive(False)
obsdone_DHYDRO = solara.reactive(False) 
mdudone_DHYDRO = solara.reactive(False) 
rundone_DHYDRO = solara.reactive(False) 


## 2) User Input

In [28]:
####################################### User Input ########################################
model_name_DHYDRO = solara.reactive("") # solara.reactive("Model_Name")
continuous_update_DHYDRO = solara.reactive(True)
model_resolution_options = ["0.05", "0.5"]
dx = solara.reactive("0.05")
# lat_min_DHYDRO = solara.reactive(0.0); lat_max_DHYDRO = solara.reactive(0.0); lon_min_DHYDRO = solara.reactive(0.0); lon_max_DHYDRO = solara.reactive(0.0)
lat_min_DHYDRO = solara.reactive(53); lat_max_DHYDRO = solara.reactive(54.5); lon_min_DHYDRO = solara.reactive(-1.0); lon_max_DHYDRO = solara.reactive(1.5) 
rectangle_DHYDRO = Rectangle(bounds=((lat_min_DHYDRO.value, lon_min_DHYDRO.value), (lat_max_DHYDRO.value, lon_max_DHYDRO.value)), color="black", fill_opacity=0) # , weight=1)
def update_rectangle():
    if rectangle_DHYDRO in current_layer_group_DHYDRO.value.layers:
        current_layer_group_DHYDRO.value.remove_layer(rectangle_DHYDRO)
    rectangle_DHYDRO.bounds = ((lat_min_DHYDRO.value, lon_min_DHYDRO.value), (lat_max_DHYDRO.value, lon_max_DHYDRO.value))
    current_layer_group_DHYDRO.value.add_layer(rectangle_DHYDRO)

# date_min_DHYDRO = solara.reactive(datetime.date(2022, 11, 1)); date_max_DHYDRO = solara.reactive(datetime.date(2022, 11, 3)); ref_date_DHYDRO = solara.reactive(datetime.date(2022, 1, 1))
date_min_DHYDRO = solara.reactive(datetime.date(2013, 12, 1)); date_max_DHYDRO = solara.reactive(datetime.date(2013, 12, 2)); ref_date_DHYDRO = solara.reactive(datetime.date(2013, 1, 1)) 


dir_output = solara.reactive("C:\\") # ("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\test_loschen")
def makedir():
    global dir_output_data
    dir_path = Path(dir_output.value)  
    os.makedirs(dir_path, exist_ok=True)
    dir_output_data = dir_path / 'data'
    os.makedirs(dir_output_data, exist_ok=True)
    userinputdone_DHYDRO.set(True)
    
overwrite = False # used for downloading of forcing data. Always set to True when changing the domain
crs = 'EPSG:4326' # coordinate reference system

In [29]:
@solara.component
def Tab_DHYDRO_User_Input():
    solara.use_effect(update_rectangle, dependencies=[lat_min_DHYDRO.value, lat_max_DHYDRO.value, lon_min_DHYDRO.value, lon_max_DHYDRO.value]) # RESET THE RECTANGLE: Model area
    
    with solara.Card("User Input", style={"width": "100%", "padding": "10px"}):
        solara.Markdown("""**Note to the user**: In this notebook we use publicly available data 
                        from Copernicus Programme of the European Union. To access this data you 
                        need to create accounts at 
                        [Copernicus Marine Service](https://data.marine.copernicus.eu/register)
                        and the [Climate Data Store](https://cds.climate.copernicus.eu/profile).
                        Do not forget to accept the CDS license agreement.""",style={"color": "inherit"})
        solara.InputText("Model Name (avoid spaces)", value=model_name_DHYDRO, continuous_update=continuous_update_DHYDRO.value)
        solara.InputText("Output directory", value=dir_output, continuous_update=True, style={"marginBottom": "20px"})
        solara.Button(label="Select/Create directory",on_click=makedir, style={"marginBottom": "20px"})

        solara.Markdown("Area of Model & Resolution:",style={"color": "inherit"})
        solara.InputFloat("Latitude minimum", value=lat_min_DHYDRO, continuous_update=True)
        solara.InputFloat("Latitude maximum", value=lat_max_DHYDRO, continuous_update=True)
        solara.InputFloat("Longitude minimum", value=lon_min_DHYDRO, continuous_update=True)
        solara.InputFloat("Longitude maximum", value=lon_max_DHYDRO, continuous_update=True)     
        solara.Select(label="Model Resolution", value=dx, values=model_resolution_options) # Model Resolution in x-direction

        solara.Markdown("Date selection:",style={"color": "inherit"})
        solara.Text("Select min date:"); solara.lab.InputDate(date_min_DHYDRO, style={"marginBottom": "20px"})
        solara.Text("Select max date:"); solara.lab.InputDate(date_max_DHYDRO, style={"marginBottom": "20px"})
        if date_max_DHYDRO.value < date_min_DHYDRO.value:
            solara.Markdown("**Warning**: The end date cannot be earlier than the start date.", 
                            style={"color": "red"})
        ref_date_DHYDRO = date_min_DHYDRO

        with solara.Row(justify="end"):     
            solara.Button(label="Go to Step 2.", on_click=lambda: selected_tab_DHYDRO.set('GridGen'),disabled=not userinputdone_DHYDRO.value)
    
# Tab_DHYDRO_User_Input()

## 3) Grid Generation, Refinement and Bathymetry

In [30]:
# if pli and net.nc files already exist:
# dir_grid = solara.reactive("C:\\") # ("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model")
dir_grid_pli = solara.reactive("C:\\")
dir_grid_netnc = solara.reactive("C:\\")

import_grid_status = solara.reactive("Idle")
def import_grid():
    global xu_grid_uds, poly_file, netfile
    import_grid_status.set("Importing")

    poly_file = os.path.join(dir_grid_pli.value)
    with open(poly_file, 'r') as file:
        lines = file.readlines()
    pli_content = [[float(value) for value in line.split()] for line in lines[2:]]
    line_geom = LineString([(lon, lat) for lon, lat in pli_content])
    gdf = gpd.GeoDataFrame(geometry=[line_geom], crs="EPSG:4326")
    gdf_json = gdf.to_json()
    imported_bnd = GeoJSON(data=json.loads(gdf_json),style={"color": "red", "weight": 2},name="Boundaries")

    netfile = os.path.join(dir_grid_netnc.value)
    xu_grid_uds = dfmt.open_partitioned_dataset(netfile)
    plt.ioff(); mpl_linecollection = line(xu_grid_uds.grid); plt.ion()
    segments = mpl_linecollection.get_segments()
    line_geometries = [LineString(segment) for segment in segments]
    gdf = gpd.GeoDataFrame(geometry=line_geometries, crs="EPSG:4326")
    gdf_json = gdf.to_json()
    import_grid_layer = GeoJSON(data=json.loads(gdf_json),style={"color": "blue", "weight": 1},name="Grid Lines")

    for layer in list(current_layer_group_DHYDRO.value.layers):
        if isinstance(layer, GeoJSON) and layer.name == "Grid Lines":
            current_layer_group_DHYDRO.value.remove_layer(layer)
        if isinstance(layer, GeoJSON) and layer.name == "Boundaries":
            current_layer_group_DHYDRO.value.remove_layer(layer)
    current_layer_group_DHYDRO.value.add_layer(import_grid_layer) 
    current_layer_group_DHYDRO.value.add_layer(imported_bnd) 

    import_grid_status.set("Done")
    gridgendone_DHYDRO.set(True) 



plot_bathy_files_status = solara.reactive("Idle")
def plot_bathymetry_files():
    plot_bathy_files_status.set("Plotting")

    bathy_data = xu_grid_uds.mesh2d_node_z.values
    colormap = linear.viridis.scale(round(np.nanmin(bathy_data),1), round(np.nanmax(bathy_data),1))
    scatter_layer = LayerGroup()
    for la, lo, ba in zip(xu_grid_uds.mesh2d_node_z.lat.values, xu_grid_uds.mesh2d_node_z.lon.values, bathy_data):
        if np.isnan(ba):
            continue
        color = colormap(ba)
        marker = CircleMarker(
            location=(la, lo),
            radius=5,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7
        )
        scatter_layer.add_layer(marker)   
    plot_bathy_files_status.set("Done")
    current_layer_group_DHYDRO.value.add_layer(scatter_layer)
    
    legend_html = widgets.HTML(value=colormap._repr_html_())
    legend_control = WidgetControl(widget=legend_html, position='bottomright')
    tab_controls_DHYDRO.value["GridGen"] = legend_control
    control_update_signal_DHYDRO.set(control_update_signal_DHYDRO.value + 1)



In [31]:
# if starting from fresh:
# Grid Generation -------------------------------------------------------------------------------------------------------------------------------------------
grid_status = solara.reactive("Idle")
grid_visible = solara.reactive(False) 
def generate_grid():
    global mk_object, grid_layer, poly_file, gdf
    grid_status.set("Generating")
    # generate spherical regular grid
    mk_object = dfmt.make_basegrid(lon_min_DHYDRO.value, lon_max_DHYDRO.value, lat_min_DHYDRO.value, lat_max_DHYDRO.value, dx=float(dx.value), dy=float(dx.value)*1/np.cos(np.radians(lat_min_DHYDRO.value)), crs=crs) # IF dy IS INPUT: dx=float(dx.value), dy=float(dy.value), crs=crs) # PREVIOUSLY (dxy): dx=float(dxy.value), dy=float(dxy.value), crs=crs)
    # transform grid data
    mesh = mk_object.mesh2d_get(); node_x = mesh.node_x; node_y = mesh.node_y; edge_nodes = mesh.edge_nodes 
    line_geometries = [LineString([(node_x[start], node_y[start]), (node_x[end], node_y[end])]) for start, end in zip(edge_nodes[::2], edge_nodes[1::2])]
    gdf = gpd.GeoDataFrame(geometry=line_geometries, crs="EPSG:4326")
    # add grid to map
    gdf_json = gdf.to_json()
    grid_layer = GeoJSON(data=json.loads(gdf_json),style={"color": "blue", "weight": 2},name="Grid Lines")
    # generate boundaries
    bnd_gdf = dfmt.generate_bndpli_cutland(mk=mk_object, res='h', buffer=0.01)
    bnd_gdf_interp = dfmt.interpolate_bndpli(bnd_gdf, res=0.03)
    # add red boundaries to map
    bnd_gdf_json = bnd_gdf_interp.to_json()
    bnd_layer = GeoJSON(data=json.loads(bnd_gdf_json),style={"color": "red", "weight": 2},name="Boundaries")
    grid_status.set("Completed")
    # return grid_layer, bnd_layer
    for layer in list(current_layer_group_DHYDRO.value.layers):
        if isinstance(layer, GeoJSON) and layer.name == "Grid Lines":
            current_layer_group_DHYDRO.value.remove_layer(layer)
        if isinstance(layer, GeoJSON) and layer.name == "Boundaries":
            current_layer_group_DHYDRO.value.remove_layer(layer)
    current_layer_group_DHYDRO.value.add_layer(grid_layer)
    current_layer_group_DHYDRO.value.add_layer(bnd_layer)
    grid_visible.set(True)
    # generate plifile
    pli_polyfile = dfmt.geodataframe_to_PolyFile(bnd_gdf_interp, name=f'{model_name_DHYDRO.value}_bnd')
    poly_file = os.path.join(dir_output.value, f'{model_name_DHYDRO.value}.pli')
    pli_polyfile.save(poly_file)


# Grid refinement -------------------------------------------------------------------------------------------------------------------------------------------
grid_refinement_options = ["300", "3000"]
min_edge_size = solara.reactive("3000")

grid_refine_export_status = solara.reactive("Idle")
def refine_export_grid():
    global xu_grid_uds, netfile, illegalcells_gdf
    grid_refine_export_status.set("InProcess")
    
    # Refine Grid --------------
    if bathy_choice.value == "Deltares":
        file_nc_bathy = "https://opendap.deltares.nl/thredds/dodsC/opendap/deltares/Delft3D/netcdf_example_files/GEBCO_2022/GEBCO_2022_coarsefac08.nc"
        data_bathy = xr.open_dataset(file_nc_bathy).elevation
    elif bathy_choice.value == "NOAA":
        file_nc_bathy = "https://www.ngdc.noaa.gov/thredds/dodsC/global/ETOPO2022/30s/30s_surface_elev_netcdf/ETOPO_2022_v1_30s_N90W180_surface.nc"
        data_bathy = xr.open_dataset(file_nc_bathy).z
    # else: # NOT IMPLEMENTED
    # subset to area of interest
    data_bathy_sel = data_bathy.sel(lon=slice(lon_min_DHYDRO.value-1, lon_max_DHYDRO.value+1), lat=slice(lat_min_DHYDRO.value-1, lat_max_DHYDRO.value+1))
    dfmt.refine_basegrid(mk=mk_object, data_bathy_sel=data_bathy_sel, min_edge_size=min_edge_size.value)
    # PLOT
    # transform grid data
    mesh = mk_object.mesh2d_get(); node_x = mesh.node_x; node_y = mesh.node_y; edge_nodes = mesh.edge_nodes 
    line_geometries = [LineString([(node_x[start], node_y[start]), (node_x[end], node_y[end])]) for start, end in zip(edge_nodes[::2], edge_nodes[1::2])]
    gdf = gpd.GeoDataFrame(geometry=line_geometries, crs="EPSG:4326")
    # add grid to map
    gdf_json = gdf.to_json()
    grid_ref_layer = GeoJSON(data=json.loads(gdf_json),style={"color": "blue", "weight": 2},name="Grid Lines Refined")

    # remove all layers
    for layer in list(current_layer_group_DHYDRO.value.layers):
        current_layer_group_DHYDRO.value.remove_layer(layer)

    # Remove Land --------------
    # remove land with GSHHS coastlines
    dfmt.meshkernel_delete_withcoastlines(mk=mk_object, res='h')
    # derive illegalcells geodataframe
    illegalcells_gdf = dfmt.meshkernel_get_illegalcells(mk=mk_object)
    # PLOT
    # transform grid data
    mesh = mk_object.mesh2d_get(); node_x = mesh.node_x; node_y = mesh.node_y; edge_nodes = mesh.edge_nodes 
    line_geometries = [LineString([(node_x[start], node_y[start]), (node_x[end], node_y[end])]) for start, end in zip(edge_nodes[::2], edge_nodes[1::2])]
    gdf = gpd.GeoDataFrame(geometry=line_geometries, crs="EPSG:4326")
    # add grid to map
    gdf_json = gdf.to_json()
    grid_wolandcells_layer = GeoJSON(data=json.loads(gdf_json),style={"color": "blue", "weight": 2},name="Grid Lines woLand")    
    current_layer_group_DHYDRO.value.add_layer(grid_wolandcells_layer) # m.value.add_layer(grid_wolandcells_layer)
    grid_visible.set(True)

    # Convert to xugrid --------
    xu_grid_uds = dfmt.meshkernel_to_UgridDataset(mk=mk_object, crs=crs)
    # interpolate bathymetry onto the grid
    data_bathy_interp = data_bathy_sel.interp(lon=xu_grid_uds.obj.mesh2d_node_x, lat=xu_grid_uds.obj.mesh2d_node_y)
    xu_grid_uds['mesh2d_node_z'] = data_bathy_interp.clip(max=10)
    xu_grid_uds.obj.mesh2d_node_z.encoding["_FillValue"] = 1e20
    # write xugrid grid to netcdf
    netfile = os.path.join(dir_output.value, f'{model_name_DHYDRO.value}_net.nc')
    xu_grid_uds.ugrid.to_netcdf(netfile)

    grid_refine_export_status.set("Done")
    

# Plot Bathymetry ---------------------------------------------------------------------------------------------------------------------------------------
bathymetry_options = ["Deltares", "NOAA", "Own data (not implemented)"]
bathy_choice = solara.reactive("Deltares")

plot_bathy_status = solara.reactive("Idle")
legend_bathy = []
def plot_bathymetry():
    global legend_bathy
    gdf_plot = xu_grid_uds.mesh2d_node_z.ugrid.to_geodataframe(name="bathy")
    plot_bathy_status.set("Plotting")
    min_bathy, max_bathy = gdf_plot["bathy"].min(), gdf_plot["bathy"].max()
    colormap = linear.viridis.scale(round(min_bathy,2), round(max_bathy,2))

    scatter_layer = LayerGroup()  # Layer for all markers
    for _, row in gdf_plot.iterrows():
        color = colormap(row["bathy"])  # Map value to color
        marker = CircleMarker(
            location=(row["mesh2d_node_y"], row["mesh2d_node_x"]),
            radius=5,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7
        )
        scatter_layer.add_layer(marker)
    
    plot_bathy_status.set("Done")
    current_layer_group_DHYDRO.value.add_layer(scatter_layer)

    legend_html = widgets.HTML(value=f"<b>Elevation</b><br>{colormap._repr_html_()}") # widgets.HTML(value=colormap._repr_html_())
    legend_control = WidgetControl(widget=legend_html, position='bottomright')
    tab_controls_DHYDRO.value["GridGen"] = legend_control
    control_update_signal_DHYDRO.set(control_update_signal_DHYDRO.value + 1)
    

# Check Orthogonality ---------------------------------------------------------------------------------------------------------------------------------------
ortho_status = solara.reactive("Idle")
def orthogonality_check():
    ortho_status.set("Checking")
    ortho = mk_object.mesh2d_get_orthogonality()
    ortho_vals = ortho.values
    ortho_vals[ortho_vals==ortho.geometry_separator] = 0
    print('mk.mesh2d_get_orthogonality()')
    print(mk_object.mesh2d_get_orthogonality().values.max())
    xu_grid_uds['ortho'] = xr.DataArray(ortho_vals, dims=xu_grid_uds.grid.edge_dimension)
    ortho_status.set("Done")
    gridgendone_DHYDRO.set(True) 


In [32]:
@solara.component
def Tab_DHYDRO_Grid():    

    with solara.lab.Tabs(): 
            with solara.lab.Tab("Import Existing"):
                with solara.Card("Grid Generation", style={"width": "100%", "padding": "10px"}):
                    solara.Markdown("Follow this in case grid data already exists (i.e. '.pli' and '_net.nc') for the model case.",style={"color": "inherit"})
                    solara.InputText("Grid file (.pli)", value=dir_grid_pli, continuous_update=True, style={"marginBottom": "20px"})
                    path_pli = Path(dir_grid_pli.value); pli_valid = False
                    if dir_grid_pli.value:
                        if not dir_grid_pli.value.endswith(".pli"):
                            solara.Error(label='Error: File must end with ".pli"', text=False, dense=True, outlined=True, icon=False)
                        elif not path_pli.exists():
                            solara.Error("File does not exist.")
                        else:
                            pli_valid = True
                    
                    solara.InputText("Grid file (_net.nc)", value=dir_grid_netnc, continuous_update=True, style={"marginBottom": "20px"})
                    path_netnc = Path(dir_grid_netnc.value); netnc_valid = False
                    if dir_grid_netnc.value:
                        if not dir_grid_netnc.value.endswith("_net.nc"):
                            solara.Error(label='Error: File must end with "_net.nc"', text=False, dense=True, outlined=True, icon=False)
                        elif not path_netnc.exists():
                            solara.Error("File does not exist.")
                        else:
                            netnc_valid = True
                    
                    solara.Markdown("**Importing:**",style={"color": "inherit"})
                    solara.Button(label="Import Grid",on_click=import_grid, continuous_update=True, disabled=not (pli_valid and netnc_valid), style={"marginBottom": "20px"})
                    if import_grid_status.value == "Importing":
                        solara.Markdown("Importing... Please wait.",style={"color": "inherit"})
                    elif import_grid_status.value == "Done":
                        solara.Markdown("Done!",style={"color": "inherit"})
                    
                    solara.Markdown("**Bathymetry:**",style={"color": "inherit"})
                    solara.Button(label="Plot Bathymetry",on_click=plot_bathymetry_files, continuous_update=True, disabled=not (pli_valid and netnc_valid), style={"marginBottom": "20px"})
                    if plot_bathy_files_status.value == "Plotting":
                        solara.Markdown("Preparing plot... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
                    elif plot_bathy_files_status.value == "Done":
                        solara.Markdown("Done!",style={"color": "inherit"})
                        
                with solara.Row(justify="end"):     
                    solara.Button(label="Go to Step 3.", on_click=lambda: selected_tab_DHYDRO.set('Bnd'),disabled=not gridgendone_DHYDRO.value)


                
            with solara.lab.Tab("Create New"):
                with solara.Card("Grid Generation", style={"width": "100%", "padding": "10px"}):
                    solara.Button(label="Generate Grid",on_click=lambda: (generate_grid(), grid_generated_DHYDRO.set(True)),continuous_update=True, style={"marginBottom": "20px"}) # solara.Button(label="Generate Grid",on_click=generate_grid, continuous_update=True)
                    if grid_status.value == "Generating":
                        solara.Markdown("Generating grid... Please wait.",style={"color": "inherit"})
                    elif grid_status.value == "Completed":
                        solara.Markdown("Grid generation completed successfully!",style={"color": "inherit"})
                    
                    solara.Markdown("**Grid Refinement:**",style={"color": "inherit"})
                    solara.Select(label="Grid Refinement", value=min_edge_size, values=grid_refinement_options)
            
                    solara.Button(label="Refine and Export Grid",on_click=lambda: (refine_export_grid(), grid_refined_DHYDRO.set(True)),continuous_update=True,disabled=not grid_generated_DHYDRO.value, style={"marginBottom": "20px"}) # solara.Button(label="Refine and Export Grid",on_click=refine_export_grid,continuous_update=True,disabled=not grid_generated_DHYDRO.value)
                    if not grid_generated_DHYDRO.value:
                        solara.Markdown("Grid needs to be generated first.",style={"color": "inherit"})
                    if grid_refine_export_status.value == "InProcess":
                        solara.Markdown("Refining grid... Please wait.",style={"color": "inherit"})
                    elif grid_refine_export_status.value == "Done":
                        solara.Markdown("Grid refinement and exportation completed successfully!",style={"color": "inherit"}) 
                    
                    solara.Markdown("**Bathymetry:**",style={"color": "inherit"})
                    solara.Markdown("""Note, if 'Own data' is chosen: You can download your own full 
                                    resolution cutout from [gebco data]( https://download.gebco.net) 
                                    (use a buffer of e.g. 1 degree).""",style={"color": "inherit"})
                    solara.Select(label="Bathymetry", value=bathy_choice, values=bathymetry_options)
                    
                    solara.Button(label="Plot Bathymetry",on_click=plot_bathymetry, continuous_update=True,disabled=not grid_refined_DHYDRO.value, style={"marginBottom": "20px"}) # solara.Button(label="Plot Bathymetry",on_click=lambda: (plot_bathymetry(), grid_refined_DHYDRO.set(True)), continuous_update=True,disabled=not grid_refined_DHYDRO.value) 
                    if not grid_refined_DHYDRO.value:
                        solara.Markdown("Grid needs to be refined first.",style={"color": "inherit"})
                    if plot_bathy_status.value == "Plotting":
                        solara.Markdown("Preparing plot... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
                    elif plot_bathy_status.value == "Done":
                        solara.Markdown("Done!",style={"color": "inherit"})   
            
                    solara.Markdown("**Check Orthogonality:**",style={"color": "inherit"})
                    solara.Button(label="Check",on_click=orthogonality_check, continuous_update=True,disabled=not grid_refined_DHYDRO.value, style={"marginBottom": "20px"}) 
                    if ortho_status.value == "Checking":
                        solara.Markdown("Checking Orthogonality... Please wait.",style={"color": "inherit"})
                    elif ortho_status.value == "Done":
                        solara.Markdown("Done!",style={"color": "inherit"})  
                        max_ortho = mk_object.mesh2d_get_orthogonality().values.max()
                        if max_ortho >= 0.4:
                            solara.Markdown(f"**Warning:** Orthogonality value is too high ({max_ortho:.2f}). Please review your grid.",
                                            style={"color": "red"})
                        else:
                            solara.Markdown(f"Orthogonality value ({max_ortho:.2f}) is below the threshold. You may proceed.",style={"color": "inherit"})

                with solara.Row(justify="end"):     
                    solara.Button(label="Go to Step 3.", on_click=lambda: selected_tab_DHYDRO.set('Bnd'),disabled=not gridgendone_DHYDRO.value)


# Tab_DHYDRO_Grid()

## 4) Boundary Conditions (tidal model and CMEMS)

In [33]:
# if _new.ext and linked files already exist:
dir_new_ext = solara.reactive("C:\\") # solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model")

add_ext_status = solara.reactive("Idle")
def add_wl_bnd(dir_new_ext):
    add_ext_status.set("Running")
    global ext_file_new
    ext_file_new = os.path.join(dir_new_ext.value)#, f'{model_name_DHYDRO.value}_new.ext')
    add_ext_status.set("Done")
    bnddone_DHYDRO.set(True) 

In [34]:
# if starting from fresh:
# Tide Model Selection -----------------------------------------------------------------------------------------------------------------------------------------
tidem_options = ["GTSMv4.1_opendap", "GTSMv4.1", "Other (not working outside Deltares)"]
tidemodel = solara.reactive("GTSMv4.1_opendap") # tidemodel: FES2014, FES2012, EOT20, GTSMv4.1, GTSMv4.1_opendap

# Generating tidal boundaries ----------------------------------------------------------------------------------------------------------------------------------
tidal_bnd_status = solara.reactive("Idle")
def tidal_bnd():
    tidal_bnd_status.set("Running")
    global ext_file_new, ext_new
    # generate new format external forcings file (.ext): initial and open boundary condition
    ext_file_new = os.path.join(dir_output.value, f'{model_name_DHYDRO.value}_new.ext')
    ext_new = hcdfm.ExtModel()
    # interpolate tidal components to boundary conditions file (.bc)
    dfmt.interpolate_tide_to_bc(ext_new=ext_new, tidemodel=tidemodel.value, file_pli=poly_file, component_list=None)
    tidal_bnd_status.set("Completed")

# download GTSM data and restructure and make bc file ----------------------------------------------------------------------------------------------------------
download_GTSM_status = solara.reactive("Idle")
def download_GTSM():
    global dir_output_data_gtsm, ds_gtsm, ext_new
    download_GTSM_status.set("Running")
    # download GTSM data and select only grid points (no coastline data or otherwise)
    dir_output_data_gtsm = os.path.join(dir_output_data, 'GTSM')
    os.makedirs(dir_output_data_gtsm, exist_ok=True)
    gtsm_era5_gpd = dfmt.ssh_catalog_subset(source='gtsm3-era5-cds')
    subset_kwargs = dict(lon_min=lon_min_DHYDRO.value, lon_max=lon_max_DHYDRO.value, lat_min=lat_min_DHYDRO.value, lat_max=lat_max_DHYDRO.value, 
                     time_min=date_min_DHYDRO.value.strftime("%Y-%m-%d"), time_max=date_max_DHYDRO.value.strftime("%Y-%m-%d"))
    gtsm_era5_gpd_sel = dfmt.ssh_catalog_subset(source='gtsm3-era5-cds', **subset_kwargs)
    bool_eur = gtsm_era5_gpd_sel['station_name'].str.startswith("id_reg_grid_eur")
    gtsm_era5_gpd_sel_grid = gtsm_era5_gpd_sel.loc[bool_eur]
    #download nc files per point
    dfmt.ssh_retrieve_data(gtsm_era5_gpd_sel_grid, dir_output_data_gtsm,  
                            time_min=date_min_DHYDRO.value.strftime("%Y-%m-%d"), time_max=date_max_DHYDRO.value.strftime("%Y-%m-%d"))

    # restructuring
    file_pattern_gtsm = os.path.join(dir_output_data_gtsm, 'gtsm3-era5-*-id_reg_grid_eur_*.nc')
    file_list_gtsm = glob.glob(file_pattern_gtsm)
    datasets_gtsm = [xr.open_dataset(file) for file in file_list_gtsm]
    for i, ds in enumerate(datasets_gtsm):
        ds = ds.expand_dims({'stations': [ds.attrs['station_name']]})
        datasets_gtsm[i] = ds
    ds_gtsm = xr.concat(datasets_gtsm, dim='stations')

    # make bc file
    ds_gtsm = ds_gtsm.set_coords(["station_x_coordinate", "station_y_coordinate"])
    wl_GTSM = ds_gtsm[['waterlevel']]
    wl_GTSM.waterlevel.attrs['units'] = 'm'
    file_pli = Path(dir_output.value, f'{model_name_DHYDRO.value}.pli')
    data_interp = dfmt.interp_hisnc_to_plipoints(data_xr_his=wl_GTSM, file_pli=file_pli, kdtree_k=4)
    data_interp = data_interp.rename({'waterlevel':'waterlevelbnd'}) 
    ForcingModel_object = dfmt.plipointsDataset_to_ForcingModel(plipointsDataset=data_interp)
    file_bc_out = file_pli.name.replace('.pli','.bc')
    ForcingModel_object.save(filepath=file_bc_out) #TODO REPORT: writing itself is fast, but takes quite a while to start writing (probably because of conversion)
    # Relocate bc file 
    source = os.path.join(os.getcwd(), file_bc_out)
    destination = os.path.join(dir_output.value, file_bc_out)
    shutil.move(source, destination)
    
    boundary_object = hcdfm.Boundary(quantity='waterlevelbnd', #the FM quantity for tide is also waterlevelbnd
                                        locationfile=file_pli,
                                        forcingfile=ForcingModel_object)
    dfmt.interpolate_grid2bnd.ext_add_boundary_object_per_polyline(ext_new=ext_new, boundary_object=boundary_object)
    ext_new.save(filepath=ext_file_new)

    download_GTSM_status.set("Completed")


# additional CMEMS xuxyadvectionvelocity boundary
# CMEMS boundaries ---------------------------------------------------------------------------------------------------------------------------------------------
cmems_bnd_status = solara.reactive("Idle")
def cmems_bnd():
    global ext_new
    cmems_bnd_status.set("Running")
    dir_output_data_cmems = os.path.join(dir_output_data, 'cmems')
    os.makedirs(dir_output_data_cmems, exist_ok=True)
    for varkey in ['uo','vo']:
        dfmt.download_CMEMS(varkey=varkey,
                            longitude_min=lon_min_DHYDRO.value-1, longitude_max=lon_max_DHYDRO.value+1, latitude_min=lat_min_DHYDRO.value-1, latitude_max=lat_max_DHYDRO.value+1,
                            date_min=date_min_DHYDRO.value.strftime("%Y-%m-%d"), date_max=date_max_DHYDRO.value.strftime("%Y-%m-%d"),
                            dir_output=dir_output_data_cmems, file_prefix='cmems_', overwrite=overwrite)        

    list_quantities = ['uxuyadvectionvelocitybnd']
    dir_pattern = os.path.join(dir_output_data_cmems,'cmems_{ncvarname}_*.nc')
    ref_date_DHYDRO = date_min_DHYDRO
    ext_new = dfmt.cmems_nc_to_bc(ext_new=ext_new,
                                refdate_str=f'minutes since {ref_date_DHYDRO.value.strftime("%Y-%m-%d")} 00:00:00 +00:00',
                                dir_output=dir_output.value,
                                list_quantities=list_quantities,
                                tstart=date_min_DHYDRO.value.strftime("%Y-%m-%d"),
                                tstop=date_max_DHYDRO.value.strftime("%Y-%m-%d"), 
                                file_pli=poly_file,
                                dir_pattern=dir_pattern)
    ext_new.save(filepath=ext_file_new)
    cmems_bnd_status.set("Completed")
    bnddone_DHYDRO.set(True) 



In [35]:
@solara.component
def Tab_DHYDRO_Boundary_Cond():

    with solara.lab.Tabs(): 
            with solara.lab.Tab("Import Existing"):
                with solara.Card("Boundary Conditions", style={"width": "100%", "padding": "10px"}):
                    solara.Markdown("Follow this in case an external forcing file ('_new.ext') exists including ALL linked necessary boundary files (i.e. '.bc' and '.pli') for the model case.",style={"color": "inherit"})
    
                    solara.Markdown("**External Forcing file:**",style={"color": "inherit"})
                    solara.InputText("Directory of the external forcing file ('_new.ext')", value=dir_new_ext, continuous_update=True, style={"marginBottom": "20px"})
                    # if dir_new_ext.value and not dir_new_ext.value.endswith("_new.ext"):
                    #     solara.Error(label='Error: File must end with "_new.ext"', text=False, dense=True, outlined=True, icon=False)
                    path_new_ext = Path(dir_new_ext.value); newext_valid = False
                    if dir_new_ext.value:
                        if not dir_new_ext.value.endswith("_new.ext"):
                            solara.Error(label='Error: File must end with "_new.ext"', text=False, dense=True, outlined=True, icon=False)
                        elif not path_new_ext.exists():
                            solara.Error("File does not exist.")
                        else:
                            newext_valid = True

                    
                    solara.Button(label="Add File",on_click=lambda: add_wl_bnd(dir_new_ext), continuous_update=True, disabled=not newext_valid)
                    if add_ext_status.value == "Done":
                        solara.Markdown(f"External Forcing file added: '{ext_file_new}'",style={"color": "inherit"})
                with solara.Row(justify="end"):     
                    solara.Button(label="Go to Step 4.", on_click=lambda: selected_tab_DHYDRO.set('Forcing'),disabled=not bnddone_DHYDRO.value)




            with solara.lab.Tab("Create New"):
                with solara.Card("Boundary Conditions", style={"width": "100%", "padding": "10px"}):
                    solara.Markdown("**Tide Model:**",style={"color": "inherit"})
                    solara.Select(label="Tide Model", value=tidemodel, values=tidem_options)
            
                    solara.Button(label="Create Tidal Boundaries",on_click=lambda: (tidal_bnd(), tidalbnddone_DHYDRO.set(True)), continuous_update=True, style={"marginBottom": "20px"})
                    if tidal_bnd_status.value == "Running":
                        solara.Markdown("**Creating...** Please wait. This takes a couple of minutes.",style={"color": "inherit"})
                    elif tidal_bnd_status.value == "Completed":
                        solara.Markdown("**Tidal Boundaries created successfully!**",style={"color": "inherit"}) 
                        
                    solara.Markdown("**Download GTSM data:**",style={"color": "inherit"})
                    solara.Markdown("Note: Download of GTSM data only implemented for European areas.",style={"color": "inherit"})
                    solara.Button(label="Download",on_click=lambda: (download_GTSM(), downloadGTSMdone_DHYDRO.set(True)), continuous_update=True, disabled=not tidalbnddone_DHYDRO.value, style={"marginBottom": "20px"}) # solara.Button(label="Download",on_click=download_GTSM, continuous_update=True)
                    if not tidalbnddone_DHYDRO.value:
                        solara.Markdown("Tidal Boundaries need to be generated first.",style={"color": "inherit"})
                    if download_GTSM_status.value == "Running":
                        solara.Markdown("**Downloading...** Please wait. This takes a couple of minutes.",style={"color": "inherit"})
                    elif download_GTSM_status.value == "Completed":
                        solara.Markdown("**GTSM data downloaded successfully!**",style={"color": "inherit"})
                    
                    solara.Markdown("**Get data from CMEMS:**",style={"color": "inherit"})
                    solara.Button(label="Download",on_click=cmems_bnd, continuous_update=True, disabled=not downloadGTSMdone_DHYDRO.value, style={"marginBottom": "20px"})
                    if not downloadGTSMdone_DHYDRO.value:
                        solara.Markdown("GTSM data needs to be downloaded first.",style={"color": "inherit"})
                    if cmems_bnd_status.value == "Running":
                        solara.Markdown("**Downloading...** Please wait. This might take a couple of minutes.",style={"color": "inherit"})
                    elif cmems_bnd_status.value == "Completed":
                        solara.Markdown("**Data downloaded and saved successfully!**",style={"color": "inherit"}) 
                with solara.Row(justify="end"):     
                    solara.Button(label="Go to Step 4.", on_click=lambda: selected_tab_DHYDRO.set('Forcing'),disabled=not bnddone_DHYDRO.value)

# Tab_DHYDRO_Boundary_Cond()

## 5) Generate CMEMS ini cond and ERA5 meteo forcing

In [36]:
# if _old.ext file (and those referred to within) already exist:
dir_old_ext = solara.reactive("C:\\") # solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model")

add_old_ext_status = solara.reactive("Idle")
def add_old_bnd(dir_old_ext):
    add_old_ext_status.set("Running")
    global ext_file_old
    ext_file_old = os.path.join(dir_old_ext.value)#, f'{model_name_DHYDRO.value}_old.ext') # HIER HIER HIER vorher _new_
    add_old_ext_status.set("Done")
    forcingdone_DHYDRO.set(True)

In [37]:
# if starting from fresh:
gen_ext_forcing_status = solara.reactive("Idle")
def gen_ext_forcing():
    global ext_file_old, ext_old
    gen_ext_forcing_status.set("Running")
    # generate old format external forcings file (.ext): spatial data
    ext_file_old = os.path.join(dir_output.value, f'{model_name_DHYDRO.value}_old.ext')
    ext_old = hcdfm.ExtOldModel()
    gen_ext_forcing_status.set("Completed")


download_save_ERA5_status = solara.reactive("Idle")
def download_ERA5():
    global varlist_list, dir_output_data_era5, ext_old
    download_save_ERA5_status.set("Running")
    # ERA5 - download spatial fields of air pressure, wind speeds and Charnock coefficient
    dir_output_data_era5 = os.path.join(dir_output_data, 'ERA5')
    os.makedirs(dir_output_data_era5, exist_ok=True)
        
    varlist_list = [['msl','u10n','v10n','chnk']]
    for varlist in varlist_list:
        for varkey in varlist:
            dfmt.download_ERA5(varkey, 
                            longitude_min=lon_min_DHYDRO.value, longitude_max=lon_max_DHYDRO.value, latitude_min=lat_min_DHYDRO.value, latitude_max=lat_max_DHYDRO.value,
                            date_min=date_min_DHYDRO.value.strftime("%Y-%m-%d"), date_max=date_max_DHYDRO.value.strftime("%Y-%m-%d"),
                            dir_output=dir_output_data_era5, overwrite=overwrite)
    download_save_ERA5_status.set("Completed")
    
    # ERA5 meteo - convert to netCDF for usage in Delft3D FM
    ext_old = dfmt.preprocess_merge_meteofiles_era5(ext_old=ext_old,
                                                    varkey_list=['msl','u10n','v10n','chnk'],# varlist_list,
                                                    dir_data=dir_output_data_era5,
                                                    dir_output=dir_output.value,
                                                    time_slice=slice(date_min_DHYDRO.value.strftime("%Y-%m-%d"), date_max_DHYDRO.value.strftime("%Y-%m-%d")))

    ext_old.save(filepath=ext_file_old) # , path_style=path_style)
    forcingdone_DHYDRO.set(True)

In [38]:
@solara.component
def Tab_DHYDRO_Forcings():

    with solara.lab.Tabs(): 
            with solara.lab.Tab("Import Existing"):
                with solara.Card("Forcings", style={"width": "100%", "padding": "10px"}):
                    solara.Markdown("Follow this in case an external forcing file ('_old.ext') exists including ALL linked necessary files for the model case.",style={"color": "inherit"})
    
                    solara.Markdown("**External Forcing file:**",style={"color": "inherit"})
                    solara.InputText("Directory of the external forcing file ('_old.ext')", value=dir_old_ext, continuous_update=True, style={"marginBottom": "20px"})
                    path_old_ext = Path(dir_old_ext.value); oldext_valid = False
                    if dir_old_ext.value:
                        if not dir_old_ext.value.endswith("_old.ext"):
                            solara.Error(label='Error: File must end with "_old.ext"', text=False, dense=True, outlined=True, icon=False)
                        elif not path_old_ext.exists():
                            solara.Error("File does not exist.")
                        else:
                            oldext_valid = True

                    solara.Button(label="Add File",on_click=lambda: add_old_bnd(dir_old_ext), continuous_update=True, disabled=not oldext_valid)
                    if add_old_ext_status.value == "Done":
                        solara.Markdown(f"External Forcing file added: '{ext_file_old}'",style={"color": "inherit"})

                with solara.Row(justify="end"):     
                    solara.Button(label="Go to Step 5.", on_click=lambda: selected_tab_DHYDRO.set('Obs'),disabled=not forcingdone_DHYDRO.value)


            with solara.lab.Tab("Create New"):
    
                with solara.Card("Forcings", style={"width": "100%", "padding": "10px"}):
                    solara.Markdown("**Generate external forcing file:**",style={"color": "inherit"})
                    solara.Button(label="Generate",on_click=lambda: (gen_ext_forcing(), genextforcingdone_DHYDRO.set(True)), continuous_update=True, style={"marginBottom": "20px"})
                    if gen_ext_forcing_status.value == "Running":
                        solara.Markdown("**Generating ...** Please wait. This might take a couple of minutes.",style={"color": "inherit"})
                    elif gen_ext_forcing_status.value == "Completed":
                        solara.Markdown("**Generation successfully!**",style={"color": "inherit"})
            
                    solara.Markdown("**Download and save ERA5 data:**",style={"color": "inherit"})
                    solara.Button(label="Download & Save",on_click=download_ERA5, continuous_update=True, disabled=not genextforcingdone_DHYDRO.value, style={"marginBottom": "20px"})
                    if not genextforcingdone_DHYDRO.value:
                        solara.Markdown("Generating external forcing file needs to be done first.",style={"color": "inherit"})
                    if download_save_ERA5_status.value == "Running":
                        solara.Markdown("**Download and Save ...** Please wait. This might take a couple of minutes.",style={"color": "inherit"})
                    elif download_save_ERA5_status.value == "Completed":
                        solara.Markdown("**Download & Save successful!**",style={"color": "inherit"}) 

                with solara.Row(justify="end"):     
                    solara.Button(label="Go to Step 5.", on_click=lambda: selected_tab_DHYDRO.set('Obs'),disabled=not forcingdone_DHYDRO.value)

# Tab_DHYDRO_Forcings()

## 6) Generate obsfile

- either input box (float) for observation location (x, y)
- draw in map (not yet implemented)

both options:
- name per selected location (x,y pair) ------>>> x, y & name as DataFrame!
- to be added to a list and all shown in map somehow

In [39]:
# if .xyn file already exists:
dir_xyn = solara.reactive("C:\\") # solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model") # 

add_xyn_status = solara.reactive("Idle")
def add_xyn(dir_xyn, size=0.1):
    add_xyn_status.set("Running")
    global file_obs
    file_obs = os.path.join(dir_xyn.value)#, f'{model_name_DHYDRO.value}_obs.xyn')


    # add on map ----------------------------------------------------
    if first_run_flag.value:
        first_run_flag.set(False)
        for layer in list(current_layer_group_DHYDRO.value.layers):
            current_layer_group_DHYDRO.value.remove_layer(layer)


    data = np.loadtxt(file_obs, usecols=(0, 1)); data = np.atleast_2d(data) #######NEU NEU NEU NEU NEU
    x = data[:, 0]; y = data[:, 1]
    
    for lat, lon in zip(y, x):
        vertical = Polyline(locations=[(lat - size, lon), (lat + size, lon)],color="red", weight=2)
        horizontal = Polyline(locations=[(lat, lon - size), (lat, lon + size)],color="red", weight=2)
        
        current_layer_group_DHYDRO.value.add_layer(horizontal)
        current_layer_group_DHYDRO.value.add_layer(vertical)
    
    add_xyn_status.set("Done")
    obsdone_DHYDRO.set(True)

In [40]:
# if starting from fresh:
obs_name = solara.reactive("Location_Name")

first_run_flag = solara.reactive(True)
def add_cross(lon, lat, size=0.1): # add on map ----------------------------------------------------
    if first_run_flag.value:
        first_run_flag.set(False)
        for layer in list(current_layer_group_DHYDRO.value.layers):
            current_layer_group_DHYDRO.value.remove_layer(layer)
            
    vertical = Polyline(
        locations=[(lat - size, lon), (lat + size, lon)],
        color="red", weight=2)
    horizontal = Polyline(
        locations=[(lat, lon - size), (lat, lon + size)],
        color="red", weight=2)

    current_layer_group_DHYDRO.value.add_layer(horizontal) 
    current_layer_group_DHYDRO.value.add_layer(vertical) 


xi = solara.reactive(0.0) # lon!
yi = solara.reactive(0.0) # lat!

obs_list = []; obs_name_list = []
formatted_locations = solara.reactive("") 
def add_loc(): # add loc given textbox inputs -------------------------------------------------------
    global obs_list, obs_name_list
    obs_list.append((xi.value, yi.value)) # (lon, lat)
    obs_name_list.append((obs_name.value))
    formatted_locations.set("; ".join(f"({round(lon, 1)}, {round(lat, 1)})" for lon, lat in obs_list))
    
    add_cross(xi.value, yi.value) # x, y --> lon, lat

    obs_name.set("Location_Name")
    xi.set(0.0) # lon!            
    yi.set(0.0) # lat!            

def clear_obs(): # clear all obs from textbox inputs -------------------------------------------------
    global obs_list
    obs_list = []
    formatted_locations.set("")
    # remove layers from map:
    layer_group = tab_layers_DHYDRO[selected_tab_DHYDRO.value]
    layers_to_remove = [layer for layer in layer_group.layers if isinstance(layer, Polyline) and layer.color == 'red']
    for layer in layers_to_remove:
        layer_group.remove_layer(layer)
    for layer in list(current_layer_group_DHYDRO.value.layers):
        current_layer_group_DHYDRO.value.remove_layer(layer)

def save_obs():
    global obs_pd, file_obs
    obs_pd = pd.DataFrame(zip(*zip(*obs_list), obs_name_list), columns=["name","x","y"])

    file_obs = os.path.join(dir_output.value, f'{model_name_DHYDRO.value}_obs.xyn')
    obs_pd.to_csv(file_obs, sep=' ', header=False, index=False, float_format='%.6f')
    
    obsdone_DHYDRO.set(True)
        

In [41]:
@solara.component
def Tab_DHYDRO_Observation():
    with solara.lab.Tabs(): 
        with solara.lab.Tab("Import Existing"):
            with solara.Card("Observations", style={"width": "100%", "padding": "10px"}):
                solara.Markdown("Follow this in case an observation file ('_obs.xyn') exists for the model case.",style={"color": "inherit"})
                #
                solara.Markdown("**Observation file:**",style={"color": "inherit"})
                solara.InputText("Directory of the observation file ('_obs.xyn')", value=dir_xyn, continuous_update=True, style={"marginBottom": "20px"})
                # if dir_xyn.value and not dir_xyn.value.endswith("_obs.xyn"):
                #     solara.Error(label='Error: File must end with "_obs.xyn"', text=False, dense=True, outlined=True, icon=False)
                path_xyn = Path(dir_xyn.value); obsxyn_valid = False
                if dir_xyn.value:
                    if not dir_xyn.value.endswith("_obs.xyn"):
                        solara.Error(label='Error: File must end with "_obs.xyn"', text=False, dense=True, outlined=True, icon=False)
                    elif not path_xyn.exists():
                        solara.Error("File does not exist.")
                    else:
                        obsxyn_valid = True

                solara.Button(label="Add File",on_click=lambda: add_xyn(dir_xyn), continuous_update=True, disabled=not obsxyn_valid)
                if add_xyn_status.value == "Done":
                    solara.Markdown(f"Observation file added: '{file_obs}'",style={"color": "inherit"})

            with solara.Row(justify="end"):     
                solara.Button(label="Go to Step 6.", on_click=lambda: selected_tab_DHYDRO.set('mdu'),disabled=not obsdone_DHYDRO.value)


        with solara.lab.Tab("Create New"):    
            with solara.Card("Observations", style={"width": "100%", "padding": "10px"}):
                solara.Markdown("Create new Observation Location:",style={"color": "inherit"})
                solara.InputText("Observation Location Name", value=obs_name, continuous_update=True)
                solara.InputFloat("Latitude", value=yi, continuous_update=True)
                solara.InputFloat("Longitude", value=xi, continuous_update=True, style={"marginBottom": "20px"})
                solara.Button("Add location", on_click=add_loc, style={"marginBottom": "20px"}) 
                solara.Markdown(f"**Locations:** {formatted_locations.value}",style={"color": "inherit"})
        
                solara.Button("Clear Locations", on_click=clear_obs, style={"marginRight": "25px","marginBottom": "20px"})
        
                solara.Button("Save Locations", on_click=save_obs)

            with solara.Row(justify="end"):     
                solara.Button(label="Go to Step 6.", on_click=lambda: selected_tab_DHYDRO.set('mdu'),disabled=not obsdone_DHYDRO.value)

# Tab_DHYDRO_Observation()

## 7) Generate mdu file

In [42]:
# settings/choices open for user
hisint = solara.reactive(600)
mapint = solara.reactive(1800)
rstint = solara.reactive(0)
statsint = solara.reactive(3600)

dimrset_folder = solara.reactive(r"p:\d-hydro\dimrset\weekly\2.28.04") 

In [43]:
# # if .mdu and other files already exists:

# gen_mdu_dimr_status_ifexist = solara.reactive("Idle")
# def generate_mdu():
#     global mdu_file
#     gen_mdu_dimr_status_ifexist.set("Running")
#     # initialize mdu file and update settings
#     mdu_file = os.path.join(dir_output.value, 'test.mdu')
#     mdu = hcdfm.FMModel()

#     # add the grid (_net.nc, network file)
#     mdu.geometry.netfile = netfile

#     # create and add drypointsfile if there are any cells generated that will result in high orthogonality
#     try: # HIER JA, WENN ILLEGALCELLS_GDF EXIST! sprich: wenn grid teil neu gemacht wurde
#         illegalcells_gdf 
#         if len(illegalcells_gdf) > 0:
#             illegalcells_polyfile = dfmt.geodataframe_to_PolyFile(illegalcells_gdf)
#             illegalcells_file = os.path.join(dir_output.value, "illegalcells.pol")
#             illegalcells_polyfile.save(illegalcells_file)
#             mdu.geometry.drypointsfile = [illegalcells_polyfile]
#     except NameError: # ELSE: überspringen?!!!!!
#         print('')

#     # add the external forcing files (.ext)
#     mdu.external_forcing.extforcefile = ext_file_old
#     mdu.external_forcing.extforcefilenew = ext_file_new

#     mdu.geometry.openboundarytolerance = 0.1
#     mdu.geometry.bedlevuni = 5 # default of -5 may cause instabilities at coastline
#     mdu.geometry.dxwuimin2d = 0.1 # improved stability in triangular network cells
#     # update numerics settings
#     mdu.numerics.izbndpos = 1 # boundary points are on network boundary
#     mdu.numerics.mintimestepbreak = 0.1 # causes instable model to crash
#     mdu.numerics.keepstbndonoutflow = 1 # s/t on outflow boundaries, should be new default
#     mdu.numerics.barocponbnd = 1 # enable baroclinic pressure gradient on open boundaries, should be new default
#     mdu.numerics.vertadvtypsal = 4 # should be new default (is currently 6)
#     mdu.numerics.vertadvtyptem = 4 # should be new default (is currently 6)
#     # update physics settings
#     mdu.physics.rhomean = 1023. # for coastal models
#     # update wind settings
#     mdu.wind.icdtyp = 4 # Charnock in case of ERA5
#     mdu.wind.cdbreakpoints = [0.025] # important if icdtyp=4, but 0.018 or 0.041 might be better. Value is overwritten by spacevarying charnock from ERA5.
#     mdu.wind.pavbnd = 101330 # for inverse barometer correction, important in case of CMEMS boundary conditions
#     # update time settings
#     mdu.time.refdate = pd.Timestamp(date_min_DHYDRO.value.strftime("%Y-%m-%d")).strftime('%Y%m%d') # pd.Timestamp(ref_date_DHYDRO.value.strftime("%Y-%m-%d")).strftime('%Y%m%d')
#     mdu.time.tunit = 'S'
#     mdu.time.dtmax = 30
#     mdu.time.startdatetime = pd.Timestamp(date_min_DHYDRO.value.strftime("%Y-%m-%d")).strftime('%Y%m%d%H%M%S')
#     mdu.time.stopdatetime = pd.Timestamp(date_max_DHYDRO.value.strftime("%Y-%m-%d")).strftime('%Y%m%d%H%M%S')
    
#     mdu.output.obsfile = [file_obs]
#     mdu.output.hisinterval = [hisint.value]
#     mdu.output.mapinterval = [mapint.value]#[86400]
#     mdu.output.rstinterval = [rstint.value]
#     mdu.output.statsinterval = [statsint.value]
    
    
#     mdu.save(mdu_file)
    
#     dfmt.make_paths_relative(mdu_file)


#     # CHAPTER 8 CODE!!! -----------------------------------------------------------------------------------------------------
#     nproc = 1 # number of processes
#     # dimrset_folder = None # previously r"p:\d-hydro\dimrset\weekly\2.28.04" # alternatively r"c:\Program Files\Deltares\Delft3D FM Suite 2024.03 HMWQ\plugins\DeltaShell.Dimr\kernels" #alternatively r"p:\d-hydro\dimrset\weekly\2.28.04"
#     dfmt.create_model_exec_files(file_mdu=mdu_file, nproc=nproc, dimrset_folder=dimrset_folder.value)
#     # remove pause at end of bat file
#     bat_loc = os.path.join(dir_output.value, 'run_parallel.bat')
#     with open(bat_loc, 'r') as file:
#         lines = file.readlines()
#     if len(lines) > 1 and 'pause' in lines[-1]:
#         lines.pop(-1)
#     with open(bat_loc, 'w') as file:
#         file.writelines(lines)
        
#     gen_mdu_dimr_status_ifexist.set("Completed")

In [44]:
# if starting from fresh:

gen_mdu_dimr_status = solara.reactive("Idle")
def generate_mdu():
    global mdu_file
    gen_mdu_dimr_status.set("Running")
    # initialize mdu file and update settings
    mdu_file = os.path.join(dir_output.value, f'{model_name_DHYDRO.value}.mdu')
    mdu = hcdfm.FMModel()

    # add the grid (_net.nc, network file)
    mdu.geometry.netfile = netfile
    mdu.geometry.openboundarytolerance = 0.1
    mdu.geometry.bedlevuni = 5 # default of -5 may cause instabilities at coastline
    mdu.geometry.dxwuimin2d = 0.1 # improved stability in triangular network cells

    # create and add drypointsfile if there are any cells generated that will result in high orthogonality
    try: # if ILLEGALCELLS_GDF EXIST!
        illegalcells_gdf 
        if len(illegalcells_gdf) > 0:
            illegalcells_polyfile = dfmt.geodataframe_to_PolyFile(illegalcells_gdf)
            illegalcells_file = os.path.join(dir_output.value, "illegalcells.pol")
            illegalcells_polyfile.save(illegalcells_file)
            mdu.geometry.drypointsfile = [illegalcells_polyfile]
    except NameError: # else do nothing here
        print('')

    # update numerics settings
    mdu.numerics.izbndpos = 1 # boundary points are on network boundary
    mdu.numerics.mintimestepbreak = 0.1 # causes instable model to crash
    mdu.numerics.keepstbndonoutflow = 1 # s/t on outflow boundaries, should be new default
    mdu.numerics.barocponbnd = 1 # enable baroclinic pressure gradient on open boundaries, should be new default
    mdu.numerics.vertadvtypsal = 4 # should be new default (is currently 6)
    mdu.numerics.vertadvtyptem = 4 # should be new default (is currently 6)

    # update physics settings
    mdu.physics.rhomean = 1023. # for coastal models

    # update wind settings
    mdu.wind.icdtyp = 4 # Charnock in case of ERA5
    mdu.wind.cdbreakpoints = [0.025] # important if icdtyp=4, but 0.018 or 0.041 might be better. Value is overwritten by spacevarying charnock from ERA5.
    mdu.wind.pavbnd = 101330 # for inverse barometer correction, important in case of CMEMS boundary conditions


    # update time settings
    mdu.time.refdate = pd.Timestamp(ref_date_DHYDRO.value.strftime("%Y-%m-%d")).strftime('%Y%m%d')
    mdu.time.tunit = 'S'
    mdu.time.dtmax = 30
    mdu.time.startdatetime = pd.Timestamp(date_min_DHYDRO.value.strftime("%Y-%m-%d")).strftime('%Y%m%d%H%M%S')
    mdu.time.stopdatetime = pd.Timestamp(date_max_DHYDRO.value.strftime("%Y-%m-%d")).strftime('%Y%m%d%H%M%S')

    # add the external forcing files (.ext)
    mdu.external_forcing.extforcefile = ext_file_old
    # mdu.external_forcing.extforcefilenew = ext_new
    try:
        mdu.external_forcing.extforcefilenew = ext_new # eigentlich ext_new; ABER: when importing existing files, ext_new is not defined
    except NameError: # so when importing existing files, follow this (leads to same result in the mdu file)
        mdu.external_forcing.extforcefilenew = ext_file_new #

    mdu.output.obsfile = [file_obs]
    mdu.output.hisinterval = [hisint.value]
    mdu.output.mapinterval = [mapint.value]#[86400]
    mdu.output.rstinterval = [rstint.value]
    mdu.output.statsinterval = [statsint.value]

    # save .mdu file
    mdu.save(mdu_file)

    # make all paths relative (might be properly implemented in https://github.com/Deltares/HYDROLIB-core/issues/532)
    dfmt.make_paths_relative(mdu_file)

    # CHAPTER 8 CODE!!! -----------------------------------------------------------------------------------------------------
    nproc = 1 # number of processes
    # dimrset_folder = None # previously r"p:\d-hydro\dimrset\weekly\2.28.04" # alternatively r"c:\Program Files\Deltares\Delft3D FM Suite 2024.03 HMWQ\plugins\DeltaShell.Dimr\kernels" #alternatively r"p:\d-hydro\dimrset\weekly\2.28.04"
    dfmt.create_model_exec_files(file_mdu=mdu_file, nproc=nproc, dimrset_folder=dimrset_folder.value)
    # remove pause at end of bat file
    bat_loc = os.path.join(dir_output.value, 'run_parallel.bat')
    with open(bat_loc, 'r') as file:
        lines = file.readlines()
    if len(lines) > 1 and 'pause' in lines[-1]:
        lines.pop(-1)
    with open(bat_loc, 'w') as file:
        file.writelines(lines)
        
    gen_mdu_dimr_status.set("Completed")
    mdudone_DHYDRO.set(True)



In [45]:
@solara.component
def Tab_DHYDRO_gen_mdu():

    with solara.Card("Generate mdu, DIMR and bat files", style={"width": "100%", "padding": "10px"}):

        solara.Text("his time interval:")
        solara.InputFloat("his Interval", value=hisint, continuous_update=True, style={"marginBottom": "20px"})
        solara.Text("map time interval:")
        solara.InputFloat("map Interval", value=mapint, continuous_update=True, style={"marginBottom": "20px"})
        solara.Text("restart time interval:")
        solara.InputFloat("rst Interval", value=rstint, continuous_update=True, style={"marginBottom": "20px"})
        solara.Text("Stats time interval:")
        solara.InputFloat("stats Interval", value=statsint, continuous_update=True, style={"marginBottom": "20px"})

        solara.Markdown("""**DIMR and bat file generation**: In order to run the model via DIMR we need a dimr_config.xml file. 
                    If you are running this notebook on a Windows platform, a *.bat file 
                    will also be created with which you can run the model directly. 
                    In order for this to work you need to update the dimrset_folder to the
                    path where the x64 and or lnx64 folder is located. Provide None if you
                    have no D-Flow FM executable available on your system.""",style={"color": "inherit"})
        solara.InputText("DIMR location", value=dimrset_folder, continuous_update=True, style={"marginBottom": "20px","marginTop": "20px"})
        
        solara.Button("Initialise files",on_click=generate_mdu)
        if gen_mdu_dimr_status.value == "Running":
            solara.Markdown("**Initialising ...** Please wait.",style={"color": "inherit"})
        elif gen_mdu_dimr_status.value == "Completed":
            solara.Markdown("**Initialisation successfully!**",style={"color": "inherit"})

    with solara.Row(justify="end"):     
        solara.Button(label="Go to Step 7.", on_click=lambda: selected_tab_DHYDRO.set('run'),disabled=not mdudone_DHYDRO.value)


# Tab_DHYDRO_gen_mdu()

## 8) Generate DIMR and bat file

In same tab as 7) (generate files)

## 9) Visualise model tree

not necessary for now, difficult to show in widget

In [46]:
# mdu_obj = hcdfm.FMModel(mdu_file)
# mdu_obj.show_tree()

## 10) Run Model

In [47]:
def check_file_size(file_path, check_interval=0.5, stabilization_checks=5):
    while not os.path.exists(file_path):
        time.sleep(check_interval)
    previous_size = -1
    stable_count = 0
    while True:
        current_size = os.path.getsize(file_path)
        if current_size == previous_size:
            stable_count += 1
            if stable_count >= stabilization_checks:
                break
        else:
            stable_count = 0
        previous_size = current_size
        time.sleep(check_interval)


model_run_status = solara.reactive("Idle")

def run_model():
    model_run_status.set("Running")
    global file_nc_his, file_nc_map

    # execute bat
    bat_loc = os.path.join(dir_output.value, 'run_parallel.bat')
    subprocess.Popen(bat_loc, shell=True, cwd=dir_output.value)
    
    file_nc_his = os.path.join(dir_output.value, f"DFM_OUTPUT_{model_name_DHYDRO.value}", f"{model_name_DHYDRO.value}_his.nc")
    file_nc_map = os.path.join(dir_output.value, f"DFM_OUTPUT_{model_name_DHYDRO.value}", f"{model_name_DHYDRO.value}_map.nc")
    check_file_size(file_nc_map)
    model_run_status.set("Completed")
    rundone_DHYDRO.set(True)

In [48]:
@solara.component
def Tab_DHYDRO_run_model():
    
    with solara.Card("Run Model", style={"width": "100%", "padding": "10px"}):
      
        solara.Button("Run Model",on_click=run_model)
        if model_run_status.value == "Running":
            solara.Markdown("**Running ...** Please wait. This might take some time, depending on model domain.",style={"color": "inherit"})
        elif model_run_status.value == "Completed":
            solara.Markdown("**Model run successful!**",style={"color": "inherit"})

# Tab_DHYDRO_run_model()     

## 11) a) Visualisation Timeseries

In [49]:
layer = None
raster_res = 0.5
umag_clim = None
scale = 15

try:
    file_nc_his
except NameError:
    file_nc_his = None
file_nc_his_new = solara.reactive("")   # user-specified path
use_manual_nc = solara.reactive(False)  # switch state
plot_fig = solara.reactive(None)

def print_timeseries():
    nc_file_to_use = None    
    if file_nc_his is not None and not use_manual_nc.value:
        nc_file_to_use = file_nc_his
    elif file_nc_his_new.value.strip() != "":
        nc_file_to_use = file_nc_his_new.value.strip()

    if not nc_file_to_use or not os.path.exists(nc_file_to_use):
        print("no netcdf file provided")
        return

    try:
        ds_his = xr.open_dataset(nc_file_to_use)
    except (TypeError, ValueError):
        ds_his = xr.open_mfdataset(nc_file_to_use)

    fig = Figure(figsize=(5, 2.5))
    ax = fig.subplots(1, 1)
    ds_his.waterlevel.plot.line(ax=ax, x='time')
    ax.legend(ds_his.station.to_series(), loc=1, fontsize=8)
    ax.grid()

    plot_fig.set(fig)



In [50]:
@solara.component
def Tab_DHYDRO_visu1():
  
    solara.use_effect(lambda: plot_fig.set(None), [use_manual_nc.value])
    
    with solara.Card("Timeseries", style={"width": "100%", "padding": "10px"}):

        # if not rundone_DHYDRO.value or file_nc_his is None:
        solara.Switch(label="Manual file path", value=use_manual_nc, style={"marginBottom": "20px"})
        if use_manual_nc.value:
            solara.InputText("Path to .nc file", value=file_nc_his_new, continuous_update=True, style={"marginBottom": "20px"})
            path_file_nc_his_new = Path(file_nc_his_new.value)
            if file_nc_his_new.value:
                if not file_nc_his_new.value.endswith("_his.nc"):
                    solara.Error(label='Error: File must end with "_his.nc"',text=False,dense=True,outlined=True,icon=False)
                elif not path_file_nc_his_new.exists():
                    solara.Error("File does not exist.")

        
        solara.Button("Create Figure",on_click=print_timeseries)
        if plot_fig.value is not None:
            solara.FigureMatplotlib(plot_fig.value)

# Tab_DHYDRO_visu1()

## 11) b) Visualisation Water levels

In [51]:
# WITHIN MAP m

try:
    file_nc_map
except NameError:
    file_nc_map = None
file_nc_map_new = solara.reactive("")      # user input path
use_manual_map1 = solara.reactive(False)    # switch state
plot_wl_map_status = solara.reactive("Idle")
# legend_wl = []
def print_wl_map():
    plot_wl_map_status.set("Plotting")
    global uds_map # legend_wl    

    nc_file_to_use = None
    if file_nc_map is not None and not use_manual_map1.value:
        nc_file_to_use = file_nc_map
    elif file_nc_map_new.value.strip():
        nc_file_to_use = file_nc_map_new.value.strip()

    if not nc_file_to_use or not os.path.exists(nc_file_to_use):
        plot_wl_map_status.set("Invalid path")
        print(f"Invalid or missing NetCDF map file: {nc_file_to_use}")
        return
    
    uds_map = dfmt.open_partitioned_dataset(nc_file_to_use)#file_nc_map)
    bool_drycells = uds_map['mesh2d_s1']==uds_map['mesh2d_flowelem_bl']
    uds_map['mesh2d_s1_filt'] = uds_map['mesh2d_s1'].where(~bool_drycells)
    gdf_wl = uds_map['mesh2d_s1_filt'].isel(time=3).ugrid.to_geodataframe(name="waterlevel")
    # remove NaNs from gdf_wl
    gdf_wl_cleaned = gdf_wl.dropna(subset=["waterlevel"])
    min_wl, max_wl = gdf_wl_cleaned["waterlevel"].min(), gdf_wl_cleaned["waterlevel"].max()
    colormap = linear.viridis.scale(round(min_wl,2), round(max_wl,2))
    sorted_wl = sorted(gdf_wl_cleaned["waterlevel"])
    
    scatter_layer = LayerGroup()
    for _, row in gdf_wl_cleaned.iterrows():
        color = colormap((row["waterlevel"]))  # Map value to color
        marker = CircleMarker(
            location=(row["mesh2d_face_y"], row["mesh2d_face_x"]),
            radius=5,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7
        )
        scatter_layer.add_layer(marker)
        
    plot_wl_map_status.set("Done")
    for layer in list(current_layer_group_DHYDRO.value.layers):
        current_layer_group_DHYDRO.value.remove_layer(layer)
    if rectangle_DHYDRO in current_layer_group_DHYDRO.value.layers: 
        current_layer_group_DHYDRO.value.remove_layer(rectangle_DHYDRO) 
    current_layer_group_DHYDRO.value.add_layer(scatter_layer)
    
    # LEGEND/COLORBAR
    legend_html = widgets.HTML(value=colormap._repr_html_())
    legend_control = WidgetControl(widget=legend_html, position='bottomright')
    tab_controls_DHYDRO.value["visu2"] = legend_control
    control_update_signal_DHYDRO.set(control_update_signal_DHYDRO.value + 1)

        

In [52]:
@solara.component
def Tab_DHYDRO_visu2():
    solara.use_effect(lambda: plot_wl_map_status.set("Idle"), [use_manual_map1.value])
    
    with solara.Card("Water levels [m]", style={"width": "100%", "padding": "10px"}):
        
        solara.Switch(label="Manual map file path", value=use_manual_map1, style={"marginBottom": "20px"})
        if use_manual_map1.value:
            solara.InputText(label="Path to .nc map file", value=file_nc_map_new, continuous_update=True, style={"marginBottom": "20px"})
            path_file_nc_map_new = Path(file_nc_map_new.value)
            if file_nc_map_new.value:
                if not file_nc_map_new.value.endswith("_map.nc"):
                    solara.Error(label='Error: File must end with "_map.nc"',text=False,dense=True,outlined=True,icon=False)
                elif not path_file_nc_map_new.exists():
                    solara.Error("File does not exist.")
        
        solara.Button("Create Map",on_click=print_wl_map)
        if plot_wl_map_status.value == "Plotting":
            solara.Markdown("Preparing plot... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
        elif plot_wl_map_status.value == "Done":
            solara.Markdown("**Done!**",style={"color": "inherit"})
            
# Tab_DHYDRO_visu2()

## 11) c) Visualisation Currents

In [53]:
try:
    file_nc_map
except NameError:
    file_nc_map = None
file_nc_map_new = solara.reactive("")      # user input path
use_manual_map2 = solara.reactive(False)    # switch state
plot_curr_map_status = solara.reactive("Idle")
legend_currents = []
def print_current_map():
    global legend_currents
    plot_curr_map_status.set("Plotting")

    nc_file_to_use = None
    if file_nc_map is not None and not use_manual_map2.value:
        nc_file_to_use = file_nc_map
    elif file_nc_map_new.value.strip():
        nc_file_to_use = file_nc_map_new.value.strip()

    if not nc_file_to_use or not os.path.exists(nc_file_to_use):
        plot_wl_map_status.set("Invalid path")
        print(f"Invalid or missing NetCDF map file: {nc_file_to_use}")
        return
    
    scaling_factor = 0.15
    uds_map = dfmt.open_partitioned_dataset(nc_file_to_use)#file_nc_map)
    uds_quiv = uds_map.isel(time=-1, mesh2d_nLayers=-2, nmesh2d_layer=-2, missing_dims='ignore')
    magn_attrs = {'long_name':'velocity magnitude', 'units':'m/s'}
    varn_ucx, varn_ucy = 'mesh2d_ucx', 'mesh2d_ucy'
    uds_quiv['magn'] = np.sqrt(uds_quiv[varn_ucx]**2+uds_quiv[varn_ucy]**2).assign_attrs(magn_attrs)
    raster_quiv = dfmt.rasterize_ugrid(uds_quiv[[varn_ucx,varn_ucy]], resolution=raster_res)

    # COLORMAP/SCATTER
    mag = uds_quiv['magn'].values
    x = uds_quiv['magn']['mesh2d_face_x'].values # mesh2d_node_x
    y = uds_quiv['magn']['mesh2d_face_y'].values # mesh2d_node_y
    min_mag = min(mag)
    max_mag = max(mag)
    colormap = linear.viridis.scale(round(min_mag,2), round(max_mag,2))
    
    scatter_layer = LayerGroup()
    for i in range(len(mag)):
        color = colormap(mag[i])  # Map value to color
        marker = CircleMarker(
            location=(y[i], x[i]),
            radius=5,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7
        )
        scatter_layer.add_layer(marker)

    # LEGEND/COLORBAR 
    legend_html = widgets.HTML(value=colormap._repr_html_())
    legend_control = WidgetControl(widget=legend_html, position='bottomright')
    tab_controls_DHYDRO.value["visu3"] = legend_control 
    control_update_signal_DHYDRO.set(control_update_signal_DHYDRO.value + 1) 
    
    # QUIVER
    x = raster_quiv.mesh2d_face_x.values
    y = raster_quiv.mesh2d_face_y.values
    u = raster_quiv.mesh2d_ucx.values # varn_ucx
    v = raster_quiv.mesh2d_ucy.values # varn_ucy

    arrow_layer = LayerGroup()
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            x_start, y_start = x[i, j], y[i, j]
            x_end = x_start + scaling_factor * u[i, j]
            y_end = y_start + scaling_factor * v[i, j]
            arrow = AntPath(
                locations=[(y_start, x_start), (y_end, x_end)],
                dash_array=[10, 20],  # Dashed pattern
                delay=1000,  # Animation speed (lower = faster)
                color="white",
                pulse_color="black"
            )
            arrow_layer.add_layer(arrow)
    for layer in list(current_layer_group_DHYDRO.value.layers):
        current_layer_group_DHYDRO.value.remove_layer(layer)
    current_layer_group_DHYDRO.value.add_layer(scatter_layer) 
    current_layer_group_DHYDRO.value.add_layer(arrow_layer)
    plot_curr_map_status.set("Done")
    

In [54]:
@solara.component
def Tab_DHYDRO_visu3():
    solara.use_effect(lambda: plot_wl_map_status.set("Idle"), [use_manual_map2.value])
    
    with solara.Card("Velocity magnitude [m/s]", style={"width": "100%", "padding": "10px"}):
        
        solara.Switch(label="Manual map file path", value=use_manual_map2, style={"marginBottom": "20px"})
        if use_manual_map2.value:
            solara.InputText(label="Path to .nc map file", value=file_nc_map_new, continuous_update=True, style={"marginBottom": "20px"})
            path_file_nc_map_new = Path(file_nc_map_new.value)
            if file_nc_map_new.value:
                if not file_nc_map_new.value.endswith("_map.nc"):
                    solara.Error(label='Error: File must end with "_map.nc"',text=False,dense=True,outlined=True,icon=False)
                elif not path_file_nc_map_new.exists():
                    solara.Error("File does not exist.")
        
        solara.Button("Create Map",on_click=print_current_map)
        if plot_curr_map_status.value == "Plotting":
            solara.Markdown("Preparing plot... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
        elif plot_curr_map_status.value == "Done":
            solara.Markdown("**Done!**",style={"color": "inherit"})


# Tab_DHYDRO_visu3()

# FIAT

## 0) Import & Settings

In [55]:

TAB_NAMES_FIAT = ["Config", "Prop","InitBuild","InspWrite"]
tab_layers_FIAT = {name: LayerGroup() for name in TAB_NAMES_FIAT}
current_layer_group_FIAT = solara.reactive(tab_layers_FIAT['Config'])
tab_controls_FIAT = solara.reactive({})
control_signals_FIAT = {name: solara.reactive(0) for name in TAB_NAMES_FIAT}
control_update_signal_FIAT = solara.reactive(0)

configdone_FIAT = solara.reactive(False) 
buildmoddone_FIAT = solara.reactive(False)

selected_tab_FIAT = solara.reactive('Config') 
current_control_FIAT = solara.reactive(None)

## 1) Configuration, Initialisation & model setup

In [56]:
model_name_FIAT = solara.reactive("") # solara.reactive("Humber")
continuous_update_FIAT = solara.reactive(True)

model_root = solara.reactive("C:\\") # ("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\FIAT\\model")


draw_control = DrawControl(rectangle={"shapeOptions": {"color": "#0000FF"}},polygon={},circle={},polyline={},circlemarker={})
draw_control.edit = False

button_fix_rect_enabled = solara.reactive(False)
draw_message = solara.reactive("")
def handle_draw(target, action, geo_json):
    button_fix_rect_enabled.set(len(draw_control.data) == 0)
    if action == "deleted":
        if len(draw_control.data) <= 2: # 2 because it is AT THE TIME OF THE CALL, not what the len is afterwards!
            draw_message.set("")
        else:
            draw_message.set("You drew too many rectangles. Only place one.")
    elif len(draw_control.data) == 0:
        draw_message.set("")
    else:
        draw_message.set("You drew too many rectangles. Only place one.")
draw_control.on_draw(handle_draw)


set_model_status = solara.reactive("Idle")
def set_model():
    global model_folder
    model_folder = Path(model_root.value) / model_name_FIAT.value  # full path to model folder
    data_catalog = (Path(os.path.abspath("")) / "Data" / "data_catalog.yml")  # path to data catalog relative to this notebook
    set_model_status.set("Done")


chosen_rectangle = solara.reactive("No rectangle set-up yet.")
def fix_model_domain():
    global mod_dom_rect, region
    mod_dom_rect = draw_control.data[0] # last_draw # last_draw has the problem problem that, if it is deleted, it is still saved as last_draw
    if mod_dom_rect and "geometry" in draw_control.last_draw:
        coords = draw_control.last_draw['geometry']['coordinates'][0][0:4]
        chosen_rectangle.set(f"Chosen rectangle between (lon, lat): ({coords[0][0]:.2f}, {coords[0][1]:.2f}) and ({coords[2][0]:.2f}, {coords[2][1]:.2f})")
    else:
        chosen_rectangle.set("No rectangle set-up yet.")

    region = gpd.GeoDataFrame(geometry=[shape(mod_dom_rect['geometry'])], crs="EPSG:4326")

    configdone_FIAT.set(True) 




# Europe, North America, Central America, South America, Asia, Africa, Oceania, or Global
continent_options = ["asia", "africa", "north america", "central and south america", "oceania", "europe", "global (not fully implemented yet)"]
selected_continent = solara.reactive("") # Europe

country_options = solara.reactive([])
continent_to_countries = {
    "asia": ["Bangladesh", "Cambodia", "China", "India", "Indonesia", "Japan", "Laos", "Pakistan", "Philippines", "Taiwan", "Thailand", "Vietnam"],
    "africa": ["Malawi" , "Mozambique", "Nigeria", "South-Africa"],
    "north america": ["USA", "Canada"],
    "central and south america": ["Argentina", "Bolivia", "Brazil", "Colombia", "El Salvador", "Guatemala", "Haiti", "Mexico", "St. Maarten", "continental"],
    "oceania": ["Australia", "New Zealand"],
    "europe": ["Belgium", "Czech Republic", "Denmark", "France", "Germany", "Hungary", "Norway", "Sweden", "Switzerland", "The Netherlands", "United Kingdom"],
    "global (not fully implemented yet)": []
}
selected_country = solara.reactive("") # United Kingdom


In [57]:
@solara.component
def Tab_FIAT_Configuration():

    tab_controls_FIAT.value["Config"] = draw_control
    solara.use_effect(lambda: country_options.set(continent_to_countries.get(selected_continent.value, [])), [selected_continent.value])
    
    with solara.Card("Configuration", style={"width": "100%", "padding": "10px"}):
        solara.Markdown("**Select Model Name:**",style={"color": "inherit"})
        solara.InputText("Model Name (avoid spaces)", value=model_name_FIAT, continuous_update=continuous_update_FIAT.value, style={"marginBottom": "20px"})

        solara.Markdown("**Select Model Path:**",style={"color": "inherit"})
        solara.InputText("Model path", value=model_root, continuous_update=True, style={"marginBottom": "20px"})
        path_model = Path(model_root.value)
        if model_root.value:
            if not path_model.exists():
                solara.Error("Path does not exist.")
        
        solara.Button("Set Path",on_click=set_model, continuous_update=True, style={"marginBottom": "20px"})
        if set_model_status.value == "Done":
            solara.Markdown("**Done!**",style={"color": "inherit"})

        solara.Markdown("**Model Domain:**",style={"color": "inherit"})
        solara.Markdown("Draw Model Domain as one rectangle in map.",style={"color": "inherit"})
        if draw_message.value:
            solara.Error(f"{draw_message.value}")
        else:
            solara.Markdown(" ")
        solara.Button("Fix Model Domain",on_click=fix_model_domain, continuous_update=True, disabled=not button_fix_rect_enabled.value, style={"marginBottom": "20px"})
        solara.Markdown(chosen_rectangle.value,style={"color": "inherit"})#f"Chosen rectangle: ({draw_control.last_draw['geometry']['coordinates'][0][0:4][0][0]:.2f}, {draw_control.last_draw['geometry']['coordinates'][0][0:4][0][1]:.2f}) and ({draw_control.last_draw['geometry']['coordinates'][0][0:4][2][0]:.2f}, {draw_control.last_draw['geometry']['coordinates'][0][0:4][2][1]:.2f})")

        solara.Markdown("**Select continent and country of the domain:**",style={"color": "inherit"})
        solara.Select(label="Continent", value=selected_continent, values=continent_options)
        solara.Select(label="Select Country", value=selected_country, values=country_options.value)
        solara.Markdown(f"Selected country: **{selected_country.value}**",style={"color": "inherit"})

    with solara.Row(justify="end"):     
        solara.Button(label="Go to Step 2.", on_click=lambda: selected_tab_FIAT.set('Prop'),disabled=not configdone_FIAT.value)


# Tab_FIAT_Configuration()

## 2) Vulnerabilty, Exposure and Max. damage

In [58]:
# Vulnerability <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# vulnerability_fname: Path to vulnerability dataset file or an entry in the data catalog that points to the vulnerability dataset file.
vul_path_or_catalog = solara.reactive("Data catalog entry") #; vul_catalog_option = solara.reactive("default")
vulnerability_fname_inp = solara.reactive("vulnerability_curves") 
vulnerability_fname = solara.reactive("")

# vulnerability_linking_fname: Path or data catalog entry of the vulnerability linking table. 
# If not provided, it is assumed that the ‘type’ in the vulnerability dataset is correct, by default None. # <<< IGNORED
vul_linking_path_or_catalog = solara.reactive("Data catalog entry")
vulnerability_linking_fname_inp = solara.reactive("vulnerability_curves_linking")
vulnerability_linking_fname = solara.reactive("")

# unit: The unit which the vulnerability index is in, by default “m” for meters.
unit = solara.reactive("m")

# Exposure <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# exposure_fname: The name of or path to the raw exposure dataset.
exp_path_or_catalog = solara.reactive("Data catalog entry")
exposure_fname_inp = solara.reactive("osm_buildings")
exposure_fname = solara.reactive("")

# exposure_type_column: The name of column in the raw dataset that specifies the object type, e.g. the occupancy type.
exp_type_option = solara.reactive("default")
exposure_type_column = solara.reactive("building")

# exposure_link_fname: The name of or path to the dataset containing the mapping of the exposure types to
# the vulnerability data, by default None.
exp_link_option = solara.reactive("Name of dataset")
exposure_link_fname_inp = solara.reactive("osm_buildings_link") 
exposure_link_fname = solara.reactive("")

# Max. damage <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# exposure_name: The name of the existing dataset.
exposure_name = solara.reactive("osm_buildings")

# exposure_type: Type of exposure corresponding with the vulnerability data, e.g. ‘damage’.
exposure_type = solara.reactive("damage")

# exposure_cost_table_fname: The name of/ path to the mapping of the costs per subtype of the exposure type, 
# e.g. ‘residential_structure’ or ‘residential_content’.
exp_cost_option = solara.reactive("Name of mapping of the costs")
exposure_cost_table_fname_inp = solara.reactive("damage_values")
exposure_cost_table_fname = solara.reactive("")

# # # # # # # # # # # # # # # # exposure_cost_link_fname: A linking table to like the present object type with the identifiers defined in the cost table.
# # # # # # # # # # # # # # # # If None, it is assumed the present object type matches the identifiers in the cost table. By default None.

# ground_flht: the height of the ground floor of all assets, in the same unit. It can also be an array 
# if assets have different ground floor heights
ground_flth = solara.reactive(0.0)
ground_elevatn = solara.reactive(0.0)
extract_method = solara.reactive("centroid")


In [59]:
@solara.component
def Tab_FIAT_Properties():
    with solara.lab.Tabs(): 
        with solara.lab.Tab("Vulnerability"):
            with solara.Card("Setup Vulnerability Parameters", style={"width": "100%", "padding": "10px"}):
                # either path (input text) or 
                # entry in data catalog pointing to vulnerability dataset file --> either default or custom (dann input text)
                # A: input text
                # B: either default or custom --> if default: see vscode inputs OR if custom: input text
                solara.Markdown("1.) Vulnerability dataset", style={"fontSize": "24px","color": "inherit"})
                solara.Text("Choose path to vulnerability dataset or data catalog entry:")
                solara.Select(label="Choose", values=["Path to vulnerability dataset", "Data catalog entry"], value=vul_path_or_catalog)
                if vul_path_or_catalog.value == "Path to vulnerability dataset":
                    solara.InputText(label="Enter Path", value=vulnerability_fname_inp, style={"marginBottom": "20px"})
                    vulnerability_fname.set(Path(vulnerability_fname_inp.value))
                    solara.Markdown(f"Chosen path: {vulnerability_fname.value}",style={"color": "inherit"})
                    if not vulnerability_fname.value.exists():
                        solara.Error("File/Path does not exist.")
                elif vul_path_or_catalog.value == "Data catalog entry":
                    solara.Text("Data catalog entry:")
                    solara.InputText(label="Enter data catalog entry", value=vulnerability_fname_inp, style={"marginBottom": "20px"})
                    vulnerability_fname.set(vulnerability_fname_inp.value)
                    solara.Markdown(f"Chosen entry: {vulnerability_fname.value}",style={"color": "inherit"})

                solara.Markdown("2.) Vulnerability linking", style={"fontSize": "24px","color": "inherit"})
                solara.Text("Choose path to vulnerability linking table or data catalog entry:")
                solara.Select(label="Choose", values=["Path to vulnerability linking dataset", "Data catalog entry"], value=vul_linking_path_or_catalog)
                if vul_linking_path_or_catalog.value == "Path to vulnerability linking dataset":
                    solara.InputText(label="Enter Path", value=vulnerability_linking_fname_inp, style={"marginBottom": "20px"})
                    vulnerability_linking_fname.set(Path(vulnerability_linking_fname_inp.value))
                    solara.Markdown(f"Chosen path: {vulnerability_linking_fname.value}",style={"color": "inherit"})
                    if not vulnerability_linking_fname.value.exists():
                        solara.Error("File/Path does not exist.")
                elif vul_linking_path_or_catalog.value == "Data catalog entry":
                    solara.Text("Data catalog entry:")
                    solara.InputText(label="Enter data catalog entry", value=vulnerability_linking_fname_inp, style={"marginBottom": "20px"})
                    vulnerability_linking_fname.set(vulnerability_linking_fname_inp.value)
                    solara.Markdown(f"Chosen entry: {vulnerability_linking_fname.value}",style={"color": "inherit"})

                solara.Markdown("3.) Unit", style={"fontSize": "24px","color": "inherit"})
                solara.Text("Choose the unit in which the vulnerability index is in.")
                solara.Select(label="Unit", values=["m"], value=unit.value)

                with solara.Row(justify="end"):  
                    solara.Button(label="Go to Step 3.", on_click=lambda: selected_tab_FIAT.set('InitBuild'))

        
        with solara.lab.Tab("Exposure"):
            with solara.Card("Setup Exposure Parameters", style={"width": "100%", "padding": "10px"}):
                
                solara.Markdown("1.) Exposure dataset", style={"fontSize": "24px","color": "inherit"})
                solara.Text("Choose path to exposure dataset or data catalog entry:")
                solara.Select(label="Choose", values=["Path to exposure dataset", "Data catalog entry"], value=exp_path_or_catalog)
                if exp_path_or_catalog.value == "Path to exposure dataset":
                    solara.InputText(label="Enter Path", value=exposure_fname_inp, style={"marginBottom": "20px"})
                    exposure_fname.set(Path(exposure_fname_inp.value))
                    solara.Markdown(f"Chosen path: {exposure_fname.value}",style={"color": "inherit"})
                    if not exposure_fname.value.exists():
                        solara.Error("File/Path does not exist.")
                elif exp_path_or_catalog.value == "Data catalog entry":
                    solara.Text("Data catalog entry:")
                    solara.InputText(label="Enter data catalog entry", value=exposure_fname_inp, style={"marginBottom": "20px"})
                    exposure_fname.set(exposure_fname_inp.value)
                    solara.Markdown(f"Chosen entry: {exposure_fname.value}",style={"color": "inherit"})
                    

                solara.Markdown("2.) Exposure type", style={"fontSize": "24px","color": "inherit"})
                solara.Text("Choose name of the column in the raw dataset, that specifies the object type.")
                solara.Select(label="Choose", values=["default", "custom"], value=exp_type_option)
                if exp_type_option.value == "default":
                    exposure_type_column.set("building")
                elif exp_type_option.value == "custom":
                    solara.InputText(label="Enter data catalog entry", value=exposure_type_column, style={"marginBottom": "20px"})

                solara.Markdown("3.) Exposure dataset containing the mapping of the exposure types", style={"fontSize": "24px","color": "inherit"})
                solara.Text("Choose name of or path to the dataset.")
                solara.Select(label="Choose", values=["Name of dataset", "Path to the dataset"], value=exp_link_option)
                if exp_link_option.value == "Path to the dataset":
                    solara.InputText(label="Enter Path", value=exposure_link_fname_inp, style={"marginBottom": "20px"})
                    exposure_link_fname.set(Path(exposure_link_fname_inp.value))
                    solara.Markdown(f"Chosen path: {exposure_link_fname.value}",style={"color": "inherit"})
                    if not exposure_link_fname.value.exists():
                        solara.Error("File/Path does not exist.")
                elif exp_link_option.value == "Name of dataset":
                    solara.Text("Name of dataset:")
                    solara.InputText(label="Enter name of dataset", value=exposure_link_fname_inp, style={"marginBottom": "20px"})
                    exposure_link_fname.set(exposure_link_fname_inp.value)
                    solara.Markdown(f"Chosen entry: {exposure_link_fname.value}",style={"color": "inherit"})

                with solara.Row(justify="end"):  
                    solara.Button(label="Go to Step 3.", on_click=lambda: selected_tab_FIAT.set('InitBuild'))
                
                
        with solara.lab.Tab("Maximum damage"):
            with solara.Card("Maximum damage", style={"width": "100%", "padding": "10px"}):
                
                solara.Markdown("1.) Exposure dataset name", style={"fontSize": "24px","color": "inherit"})
                solara.Select(label="Choose", values=["osm_buildings"], value=exposure_name) # OR INPUTTEXT

                solara.Markdown("2.) Type of Exposure corresponding with the Vulnerability data", style={"fontSize": "24px","color": "inherit"})
                solara.Select(label="Choose", values=["damage"], value=exposure_type)

                solara.Markdown("3.) Costs per subtype of exposire type", style={"fontSize": "24px","color": "inherit"})
                solara.Text("Choose name of or path to the dataset.")
                solara.Select(label="Choose", values=["Name of mapping of the costs", "Path to the mapping of the costs"], value=exp_cost_option)
                if exp_cost_option.value == "Path to the mapping of the costs":
                    solara.InputText(label="Enter Path", value=exposure_cost_table_fname_inp, style={"marginBottom": "20px"})
                    exposure_cost_table_fname.set(Path(exposure_cost_table_fname_inp.value))
                    solara.Markdown(f"Chosen path: {exposure_cost_table_fname.value}",style={"color": "inherit"})
                    if not exposure_cost_table_fname.value.exists():
                        solara.Error("File/Path does not exist.")
                elif exp_cost_option.value == "Name of mapping of the costs":
                    solara.Text("Name of dataset:")
                    solara.InputText(label="Enter name of dataset", value=exposure_cost_table_fname_inp, style={"marginBottom": "20px"})
                    exposure_cost_table_fname.set(exposure_cost_table_fname_inp.value)
                    solara.Markdown(f"Chosen entry: {exposure_cost_table_fname.value}",style={"color": "inherit"})

                # # THIS IS NOT IN THE NOTEBOOK!
                # solara.Markdown("4.) Linked Table, objects and costs", style={"fontSize": "24px"})
                # # exposure_cost_link_fname
                solara.Markdown("4.) Heights of gound floor and elevation", style={"fontSize": "24px","color": "inherit"})
                solara.InputFloat("Enter the height of the ground floor of all assets: ", value=ground_flth, continuous_update=True, style={"marginBottom": "20px"})
                solara.InputFloat("Enter the height of the ground elevation: ", value=ground_elevatn, continuous_update=True, style={"marginBottom": "20px"})
                solara.Select(label="Choose extract method: ", values=["centroid"], value=extract_method)

                with solara.Row(justify="end"):  
                    solara.Button(label="Go to Step 3.", on_click=lambda: selected_tab_FIAT.set('InitBuild'))

# Tab_FIAT_Properties()

## 3) Initialising & Build Model

In [60]:
# implement Step 2 & 3

init_model_status = solara.reactive(False)
init_model_text_status = solara.reactive("Idle")
def init_model():
    global fiat
    init_model_text_status.set("Running")
    if model_folder.exists():
        shutil.rmtree(model_folder)
    build_data = fetch_data(data="build-data")
    fiat = FIATModel(root=model_root.value, data_libs=Path(build_data,"data_catalog.yml"), mode='w+')
    init_model_status.set(True)
    init_model_text_status.set("Done")


build_model_status = solara.reactive("Idle")
def build_model():
    build_model_status.set("Running")
    fiat.setup_region(region=region)
    fiat.vulnerability.setup(vulnerability_fname=vulnerability_fname.value,
                             vulnerability_linking_fname=vulnerability_linking_fname.value,
                             unit=unit.value,
                             continent=selected_continent.value
                             )
    fiat.exposure_geoms.setup(exposure_fname=exposure_fname.value,
                          exposure_type_column=exposure_type_column.value,
                          exposure_link_fname=exposure_link_fname.value
                          ) 
    fiat.exposure_geoms.setup_max_damage(exposure_name=exposure_name.value,
                                     exposure_type=exposure_type.value,
                                     exposure_cost_table_fname=exposure_cost_table_fname.value,
                                     country=selected_country.value
                                     )
    build_model_status.set("Done")
    buildmoddone_FIAT.set(True) 



In [61]:
@solara.component
def Tab_FIAT_Init_Build():
    solara.Button("Initialise Model",on_click=init_model, continuous_update=True, style={"marginBottom": "20px"})
    if init_model_text_status.value == "Running":
        solara.Markdown("Model is being initalised... Please wait.",style={"color": "inherit"})
    elif init_model_text_status.value == "Done":
        solara.Markdown("**Model is initialised!**",style={"color": "inherit"})
    else:
        solara.Markdown("")
    
    solara.Button("Build Model",on_click=build_model, continuous_update=True, disabled=not init_model_status.value, style={"marginBottom": "20px"})
    if build_model_status.value == "Running":
        solara.Markdown("Model is being built... Please wait. This might take a couple of minutes.",style={"color": "inherit"})
    elif build_model_status.value == "Done":
        solara.Markdown("**Done, model is built.**",style={"color": "inherit"})
    else:
        solara.Markdown("")

    with solara.Row(justify="end"):     
        solara.Button(label="Go to Step 4.", on_click=lambda: selected_tab_FIAT.set('InspWrite'),disabled=not buildmoddone_FIAT.value)
    

# Tab_FIAT_Init_Build()

## 4) Inspect and write Model

In [62]:
insp_model_status = solara.reactive("Idle")
def inspect_model():
    insp_model_status.set("Running")
    def gdf_to_geojson_layer(gdf, name=None, style=None, style_callback=None):
        data = json.loads(gdf.to_json())
        kwargs = {"data": data,"name": name or "Layer"}
        if style is not None:
            kwargs["style"] = style
        if callable(style_callback):
            kwargs["style_callback"] = style_callback
        return GeoJSON(**kwargs)
 
    def style_by_object_type(feature):
        global color_map
        obj = feature["properties"].get("object_type")
        color_map = {"residential": "#9EDAE5",
            "commercial": "#1F77B4",
            "industrial": "#8C564B"}
        color = color_map.get(obj, "#8c564b")
        return {"color": color,"fillColor": color,"fillOpacity": 0.2,"weight": 1}

    gdf = fiat.exposure_geoms.data["osm_buildings"]
    region_layer = gdf_to_geojson_layer(region, name="Region",style={"color": "black", "fillColor": "transparent", "fillOpacity": 0.0, "weight": 1})
    buildings_layer = gdf_to_geojson_layer(gdf, name="object_type", style_callback=style_by_object_type)
    
    if region_layer in current_layer_group_FIAT.value.layers:
        current_layer_group_FIAT.value.remove_layer(region_layer)
    current_layer_group_FIAT.value.add_layer(region_layer)
    if buildings_layer in current_layer_group_FIAT.value.layers:
        current_layer_group_FIAT.value.remove_layer(buildings_layer)
    current_layer_group_FIAT.value.add_layer(buildings_layer)

    # LEGEND
    legend_items = []
    for label, color in color_map.items():
        item = widgets.HBox([widgets.HTML(value=f"""<div style="width:14px;height:14px;background:{color};border:1px solid #333;margin-right:6px;"></div>"""),widgets.Label(label),])
        legend_items.append(item)
    legend_box = widgets.VBox([widgets.HTML("<b>object_type</b>"),*legend_items])
    legend_control = WidgetControl(widget=legend_box, position="bottomright")
    tab_controls_FIAT.set({**tab_controls_FIAT.value, "InspWrite": legend_control}) # tab_controls.value["InspWrite"] = legend_control
    control_signals_FIAT["InspWrite"].set(control_signals_FIAT["InspWrite"].value + 1) 

    insp_model_status.set("Done")
    


write_model_status = solara.reactive("Idle")
def write_model():
    write_model_status.set("Running")
    fiat.write()
    write_model_status.set("Done")


In [63]:
@solara.component
def Tab_FIAT_Insp_Write():
    solara.Button("Inspect Model",on_click=inspect_model, continuous_update=True, style={"marginBottom": "20px"})
    if insp_model_status.value == "Running":
        solara.Markdown("Model is being inspected... Please wait.",style={"color": "inherit"})
    elif insp_model_status.value == "Done":
        solara.Markdown("**Done!**",style={"color": "inherit"})
    else:
        solara.Markdown("")

    solara.Button("Write Model",on_click=write_model, continuous_update=True, style={"marginBottom": "20px"})
    if write_model_status.value == "Running":
        solara.Markdown("Model is being written... Please wait.",style={"color": "inherit"})
    elif write_model_status.value == "Done":
        solara.Markdown("**Model is written!**",style={"color": "inherit"})
    else:
        solara.Markdown("")


# Tab_FIAT_Insp_Write()

# FINAL

In [64]:
############## SETTINGS TO EASE CODING/IMPLEMENTATION/...; TO BE REMOVED LATER!!!

###################################### SFINCS ######################################
model_name_DHYDRO = solara.reactive("Humber_delta") 
dir_output = solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model")

# if pli und netnc already exist
dir_grid_pli = solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model\\Humber_delta.pli")
dir_grid_netnc = solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model\\Humber_delta_net.nc")

# if _new.ext and linked files already exist:
dir_new_ext = solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model\\Humber_delta_new.ext")

# if _old.ext file (and those referred to within) already exist:
dir_old_ext =  solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model\\Humber_delta_old.ext")

# if .xyn file already exists:
dir_xyn = solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\model\\Humber_delta_obs.xyn") 

modeldomaindone_SFINCS.set("Done")


###################################### FIAT ######################################
model_root = solara.reactive("C:\\Users\\santjer\\OneDrive - Stichting Deltares\\Documents\\IRISCC\\FIAT\\model")
selected_continent = solara.reactive("europe") 
selected_country = solara.reactive("United Kingdom") # United Kingdom

In [65]:
######################################## SFINCS ########################################
NAME_TO_INDEX_SFINCS = {name: i for i, name in enumerate(TAB_NAMES_SFINCS)}
def idx_from_name_SFINCS(name: str) -> int:
    return NAME_TO_INDEX_SFINCS.get(name, 0)
def name_from_idx_SFINCS(i: int) -> str:
    if 0 <= i < len(TAB_NAMES_SFINCS):
        return TAB_NAMES_SFINCS[i]
    return TAB_NAMES_SFINCS[0]

@solara.component
def Sfincs_Settings():
    # solara.use_effect(lambda: selected_tab_SFINCS.set(selected_tab_SFINCS.value), [selected_tab_SFINCS.value])

    def update_map_layers_and_controls_SFINCS():
        for layer in list(m_SFINCS.layers[1:]):
            m_SFINCS.remove_layer(layer)
        # if selected_tab_SFINCS.value == 'model':
        #     elev_layers_copy = LayerGroup(layers=list(tab_layers_SFINCS["rInflP"].layers))
        #     current_layer_group_SFINCS.set(elev_layers_copy)
        #     tab_layers_SFINCS["model"] = elev_layers_copy
        #     wlbnd_layer = show_wlbnd()
        #     current_layer_group_SFINCS.value.add_layer(wlbnd_layer)
        # else:
        #     current_layer_group_SFINCS.set(tab_layers_SFINCS[selected_tab_SFINCS.value])
        current_layer_group_SFINCS.set(tab_layers_SFINCS[selected_tab_SFINCS.value])
        m_SFINCS.add_layer(current_layer_group_SFINCS.value)
        # controls
        if current_control_SFINCS.value is not None and current_control_SFINCS.value in m_SFINCS.controls:
            m_SFINCS.remove_control(current_control_SFINCS.value)
        if selected_tab_SFINCS.value in tab_controls_SFINCS.value:
            control = tab_controls_SFINCS.value[selected_tab_SFINCS.value]
            if control not in m_SFINCS.controls:
                m_SFINCS.add_control(control)
            current_control_SFINCS.set(control)
        else:
            current_control_SFINCS.set(None)
    solara.use_effect(update_map_layers_and_controls_SFINCS, [selected_tab_SFINCS.value, control_update_signal_SFINCS.value])# control_signals_SFINCS[selected_tab_SFINCS.value].value]) 


    with solara.Column(style={"width": "100%", "align-items": "center"}):
        current_index_SFINCS = idx_from_name_SFINCS(selected_tab_SFINCS.value)
        def on_tab_change_SFINCS(new_index_SFINCS: int):
            selected_tab_SFINCS.set(name_from_idx_SFINCS(new_index_SFINCS))

        with solara.lab.Tabs(value=current_index_SFINCS, on_value=on_tab_change_SFINCS, vertical=True):
            with solara.lab.Tab(tab_children="1. Configuration"):
                Tab_SFINCS_Configuration()
            with solara.lab.Tab(tab_children="2. Model Domain"):
                if not configdone_SFINCS.value:
                    solara.Markdown("**Model needs to be configurated first.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_Model_Domain()
            with solara.lab.Tab(tab_children="3. Elevation"):
                if not modeldomaindone_SFINCS.value:
                    solara.Markdown("**Model domain needs to be set-up first.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_Elevation()
            with solara.lab.Tab(tab_children="4. Active cells"):
                if not elevationdone_SFINCS.value:
                    solara.Markdown("**Elevation data needs to be determined first.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_Active_Cells()
            with solara.lab.Tab(tab_children="5. Waterlevel bound"):
                if not activecellsdone_SFINCS.value:
                    solara.Markdown("**Active cells need to be determined first.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_Waterlevel_Bound()
            with solara.lab.Tab(tab_children="6. River Infow Points"):
                if not waterlevelbnddone_SFINCS.value:
                    solara.Markdown("**Waterlevel bounds need to be determined first.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_River_Inflow_Points()
            with solara.lab.Tab(tab_children="7. Land Roughness & Subgrid"):
                if not riverinflowdone_SFINCS.value:
                    solara.Markdown("**River Inflow Points need to be determined first.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_Subgrid()
            with solara.lab.Tab(tab_children="8. Observation Stations"):
                if not roughnesssubgriddone_SFINCS.value:
                    solara.Markdown("**First follow the tab 'Land Roughness & Subgrid'.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_Observations()
            with solara.lab.Tab(tab_children="9. Forcings"):
                if not obsdone_SFINCS.value:
                    solara.Markdown("**Set observation stations first.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_setup_forcings()
            with solara.lab.Tab(tab_children="10. Show and Write Model"):
                if not forcingdone_SFINCS.value:
                    solara.Markdown("**Set forcings first.**",style={"color": "inherit"})
                else:
                    Tab_SFINCS_show_write_model()


@solara.component
def SfincsApp():
    with solara.Columns():
        with solara.Column(style={"width": "70%", "min-width": "650px"}):
            display(m_SFINCS)
        with solara.Column(style={"width": "30%", "min-width": "500px"}):
            Sfincs_Settings()


######################################## DHYDRO ########################################
NAME_TO_INDEX_DHYDRO = {name: i for i, name in enumerate(TAB_NAMES_DHYDRO)}
def idx_from_name_DHYDRO(name: str) -> int:
    return NAME_TO_INDEX_DHYDRO.get(name, 0)
def name_from_idx_DHYDRO(i: int) -> str:
    if 0 <= i < len(TAB_NAMES_DHYDRO):
        return TAB_NAMES_DHYDRO[i]
    return TAB_NAMES_DHYDRO[0]

@solara.component
def DHydro_Settings():
    solara.use_effect(lambda: selected_tab_DHYDRO.set(selected_tab_DHYDRO.value), [selected_tab_DHYDRO.value])
    def update_map_layers_and_controls_DHYDRO():
        for layer in list(m_DHYDRO.layers[1:]):
            m_DHYDRO.remove_layer(layer)
        current_layer_group_DHYDRO.set(tab_layers_DHYDRO[selected_tab_DHYDRO.value])
        m_DHYDRO.add_layer(current_layer_group_DHYDRO.value)
        if current_control_DHYDRO.value is not None and current_control_DHYDRO.value in m_DHYDRO.controls:
            m_DHYDRO.remove_control(current_control_DHYDRO.value)
        if selected_tab_DHYDRO.value in tab_controls_DHYDRO.value:
            control = tab_controls_DHYDRO.value[selected_tab_DHYDRO.value]
            if control not in m_DHYDRO.controls:
                m_DHYDRO.add_control(control)
            current_control_DHYDRO.set(control)
        else:
            current_control_DHYDRO.set(None)
    solara.use_effect(update_map_layers_and_controls_DHYDRO, [selected_tab_DHYDRO.value, control_update_signal_DHYDRO.value])

    with solara.Column(style={"width": "100%", "align-items": "center"}):
        current_index_DHYDRO = idx_from_name_DHYDRO(selected_tab_DHYDRO.value)
        def on_tab_change_DHYDRO(new_index_DHYDRO: int):
            selected_tab_DHYDRO.set(name_from_idx_DHYDRO(new_index_DHYDRO))

        with solara.lab.Tabs(value=current_index_DHYDRO, on_value=on_tab_change_DHYDRO,vertical=True):
            with solara.lab.Tab(tab_children="1. User Input"):
                Tab_DHYDRO_User_Input()
            with solara.lab.Tab(tab_children="2. Grid Generation"):
                if not userinputdone_DHYDRO.value:
                    solara.Markdown("**User input needs to be set-up first.**",style={"color": "inherit"})
                else:
                    Tab_DHYDRO_Grid()
            with solara.lab.Tab(tab_children="3. Boundary Conditions"):
                if not userinputdone_DHYDRO.value:
                    solara.Markdown("**Grid needs to be generated first.**",style={"color": "inherit"})
                else:
                    Tab_DHYDRO_Boundary_Cond()
            with solara.lab.Tab(tab_children="4. Forcings"):
                if not userinputdone_DHYDRO.value:
                    solara.Markdown("**Boundary conditions need to be set-up first.**",style={"color": "inherit"})
                else:
                    Tab_DHYDRO_Forcings()
            with solara.lab.Tab(tab_children="5. Observations"):
                if not userinputdone_DHYDRO.value:
                    solara.Markdown("**Forcing files need to be generated/implemented first.**",style={"color": "inherit"})
                else:
                    Tab_DHYDRO_Observation()
            with solara.lab.Tab(tab_children="6. Generate Files"):
                if not userinputdone_DHYDRO.value:
                    solara.Markdown("**Observation points need to be added first.**",style={"color": "inherit"})
                else:
                    Tab_DHYDRO_gen_mdu()
            with solara.lab.Tab(tab_children="7. RUN MODEL"):
                if not userinputdone_DHYDRO.value:
                    solara.Markdown("**Running/Input files needs to be generated first.**",style={"color": "inherit"})
                else:
                    Tab_DHYDRO_run_model()
            with solara.lab.Tab(tab_children="8. Timeseries"):
                Tab_DHYDRO_visu1()
            with solara.lab.Tab(tab_children="9. Water levels"):
                Tab_DHYDRO_visu2()
            with solara.lab.Tab(tab_children="10. Currents"):
                Tab_DHYDRO_visu3()


@solara.component
def DHydroApp():
    with solara.Columns():
        with solara.Column(style={"width": "70%", "min-width": "650px"}):
            display(m_DHYDRO)
        with solara.Column(style={"width": "30%", "min-width": "500px"}):
            DHydro_Settings()


# ######################################## FIAT ########################################
NAME_TO_INDEX_FIAT = {name: i for i, name in enumerate(TAB_NAMES_FIAT)}
def idx_from_name_FIAT(name: str) -> int:
    return NAME_TO_INDEX_FIAT.get(name, 0)
def name_from_idx_FIAT(i: int) -> str:
    if 0 <= i < len(TAB_NAMES_FIAT):
        return TAB_NAMES_FIAT[i]
    return TAB_NAMES_FIAT[0]

@solara.component
def Fiat_Settings():
    solara.use_effect(lambda: selected_tab_FIAT.set(selected_tab_FIAT.value), [selected_tab_FIAT.value])
    def update_map_layers_and_controls_FIAT():
        for layer in list(m_FIAT.layers[1:]):
            m_FIAT.remove_layer(layer)
        current_layer_group_FIAT.set(tab_layers_FIAT[selected_tab_FIAT.value])
        m_FIAT.add_layer(current_layer_group_FIAT.value)
        if current_control_FIAT.value is not None and current_control_FIAT.value in m_FIAT.controls:
            m_FIAT.remove_control(current_control_FIAT.value)
        if selected_tab_FIAT.value in tab_controls_FIAT.value:
            control = tab_controls_FIAT.value[selected_tab_FIAT.value]
            if control not in m_FIAT.controls:
                m_FIAT.add_control(control)
            current_control_FIAT.set(control)
        else:
            current_control_FIAT.set(None)
    solara.use_effect(update_map_layers_and_controls_FIAT, [selected_tab_FIAT.value, control_update_signal_FIAT.value])

    with solara.Column(style={"width": "100%", "align-items": "center"}):
        current_index_FIAT = idx_from_name_FIAT(selected_tab_FIAT.value)
        def on_tab_change_FIAT(new_index_FIAT: int):
            selected_tab_FIAT.set(name_from_idx_FIAT(new_index_FIAT))

        with solara.lab.Tabs(value=current_index_FIAT, on_value=on_tab_change_FIAT,vertical=True):
            with solara.lab.Tab(tab_children="1. Configuration"):
                Tab_FIAT_Configuration()
            with solara.lab.Tab(tab_children="2. Specification of Properties"):
                if not configdone_FIAT.value:
                    solara.Markdown("**Model needs to be configurated first.**",style={"color": "inherit"})
                else:
                    Tab_FIAT_Properties()
            with solara.lab.Tab(tab_children="3. Initialise and Build Model"):
                if not configdone_FIAT.value:
                    solara.Markdown("**Model needs to be configurated first.**",style={"color": "inherit"})
                else:
                    Tab_FIAT_Init_Build()
            with solara.lab.Tab(tab_children="4. Inspect and Write Model"):
                if not buildmoddone_FIAT.value:
                    solara.Markdown("**Initialise and build Model first.**",style={"color": "inherit"})
                else:
                    Tab_FIAT_Insp_Write()


@solara.component
def FiatApp():
    with solara.Columns():
        with solara.Column(style={"width": "70%", "min-width": "650px"}):
            display(m_FIAT)
        with solara.Column(style={"width": "30%", "min-width": "500px"}):
            Fiat_Settings()



In [66]:
######################################## FINAL ########################################

@solara.component
def Page():
    with solara.lab.Tabs(background_color="primary", dark=True):

        # SFINCS
        with solara.lab.Tab(label="SFINCS",tab_children=[solara.v.Img(
                    src="https://repository-images.githubusercontent.com/507055779/75f8090f-00b6-4781-a61d-8c9baa3a19ea",
                    width="20px",height="20px",contain=True,style_="margin-left: 8px; margin-right: 8px;")]):
            SfincsApp()


        # D-HYDRO
        with solara.lab.Tab(label="D-HYDRO",tab_children=[solara.v.Img( # "D-HYDRO", icon_name="mdi-water"):
                    src="https://oss.deltares.nl/image/layout_set_logo?img_id=4215092&t=1769253880864",
                    width="20px",height="20px",contain=True,style_="margin-left: 8px; margin-right: 8px;")]): 
            DHydroApp()
        

        # FIAT
        with solara.lab.Tab(label="FIAT",tab_children=[solara.v.Img(
                    src="https://deltares.github.io/Delft-FIAT/stable/_static/fiat.svg",
                    width="20px",height="20px",contain=True,style_="margin-left: 8px; margin-right: 8px;")]):
            FiatApp()

Page()

Cannot show ipywidgets in text

In [67]:
# print(selected_tab_SFINCS.value)
# print(control_signals_SFINCS)
# print(selected_tab_DHYDRO.value)

# x